# Notebook OSeMOSYS — standalone

Notebook **autocontenido** que ejecuta el flujo OSeMOSYS sin importar funciones del proyecto.

Define inline:
- helpers SAND → CSV
- AbstractModel completo (sets, params, vars, constraints, objective)
- build_instance via Pyomo DataPortal
- solver con fallback Gurobi → HiGHS → GLPK
- diagnóstico de infactibilidad

Funciona desde:
- **Excel SAND** (hoja `Parameters`)
- **Carpeta de CSVs** ya procesados

Probado con `29_04_26_SAND_INTEGRADO_CN_V2 _RM__Parameters_SAND.xlsx`.

## 1. Configuración

Edita las rutas según tu setup. Lo único que el notebook necesita del exterior:

- una **fuente de datos** (Excel SAND **o** carpeta de CSVs ya procesados),
- opcionalmente, una **licencia Gurobi**.

Si no tienes licencia Gurobi válida, el solver caerá automáticamente a HiGHS o GLPK.

In [1]:
import os
from pathlib import Path

# === FUENTE DE DATOS (activa UNA, deja la otra en None) ====================
EXCEL_PATH: Path | None = Path("/Users/davidbedoya0/Downloads/29_04_26_SAND_INTEGRADO_CN_V2 _RM__Parameters_SAND.xlsx")
CSV_DIR_IN: Path | None = None        # alternativa: ruta a carpeta CSV ya procesada

# === DESTINO DE CSVs (si se usa Excel) ====================================
CSV_DIR_OUT: Path = Path.cwd() / "tmp_osemosys_csvs"

# === LICENCIA GUROBI (opcional) ===========================================
# Si tienes WLS o named-user, apunta a tu .lic. Si no, deja como None.
GUROBI_LIC_FILE: Path | None = Path("/Users/davidbedoya0/Documents/UPME/Demanda/Repositorio/OSEMOSYS_FINAL_2/Osemosys_UPME/secrets/gurobi.lic")
if GUROBI_LIC_FILE is not None and GUROBI_LIC_FILE.is_file():
    os.environ["GRB_LICENSE_FILE"] = str(GUROBI_LIC_FILE)

# Sanity check
assert (EXCEL_PATH is None) ^ (CSV_DIR_IN is None), "Activa exactamente una fuente: EXCEL_PATH o CSV_DIR_IN"

print("EXCEL_PATH       :", EXCEL_PATH)
print("CSV_DIR_IN       :", CSV_DIR_IN)
print("CSV_DIR_OUT      :", CSV_DIR_OUT)
print("GRB_LICENSE_FILE :", os.environ.get("GRB_LICENSE_FILE", "(no set)"))

EXCEL_PATH       : /Users/davidbedoya0/Downloads/29_04_26_SAND_INTEGRADO_CN_V2 _RM__Parameters_SAND.xlsx
CSV_DIR_IN       : None
CSV_DIR_OUT      : /Users/davidbedoya0/Documents/UPME/Demanda/Repositorio/OSEMOSYS_FINAL_2/Osemosys_UPME/backend/app/simulation/tmp_osemosys_csvs
GRB_LICENSE_FILE : /Users/davidbedoya0/Documents/UPME/Demanda/Repositorio/OSEMOSYS_FINAL_2/Osemosys_UPME/secrets/gurobi.lic


## 2. Helpers SAND → CSV

Funciones para convertir un Excel SAND (hoja `Parameters`) a la estructura de CSVs que consume Pyomo `DataPortal`.

- `SAND_to_CSV`: vuelca un parámetro a su CSV (con manejo especial para `CapacityFactor` que promedia timeslices).
- `SAND_SETS_to_CSV`: deriva sets (REGION, TECHNOLOGY, FUEL, MODE, EMISSION, TIMESLICE, YEAR) desde filas de YearSplit, EmissionActivityRatio, OutputActivityRatio y CapacityToActivityUnit.
- `completar_Matrix_*`: cross-join contra todos los sets para parámetros de actividad/costo/emisión.
- `process_and_save_emission_ratios`: ajusta EmissionActivityRatio multiplicando por InputActivityRatio (factor de emisión a la entrada).
- `filter_params_by_sets`: filtra filas cuyos índices no están en los sets.

In [2]:
import os
import itertools
from itertools import product
import numpy as np
import pandas as pd


def SAND_to_CSV(df, param, path_csv, div):
    df_param = df[df["Parameter"] == param].dropna(axis=1)
    year = df_param.columns[df_param.columns.to_series().apply(pd.to_numeric, errors="coerce").notna()]
    sets = df_param.columns[~df_param.columns.isin(year) & (df_param.columns != "Parameter")].tolist()

    if "Time indipendent variables" in sets:
        sets.remove("Time indipendent variables")
        df_TUPLES = df_param.drop(["Parameter"], axis=1).rename(columns={"Time indipendent variables": "VALUE"})
    elif "TIMESLICE" in sets:
        df_param = df_param.reset_index(drop=True)
        df_param_sub = df_param[df_param.index % div == 0].reset_index(drop=True)
        df_param_sub = df_param_sub.set_index(sets).drop(columns="Parameter")

        if param == "CapacityFactor":
            df_param_sub = df_param_sub.reset_index()
            df_agg = df_param.drop(columns=["Parameter"] + sets)
            df_agg["index_col"] = df_agg.index // div
            df_agg = df_agg.groupby("index_col", as_index=False).mean()
            for y in year:
                df_param_sub[y] = df_agg[y]
            df_param_indexed = df_param_sub.set_index(sets)
            index_product = list(product(df_param_indexed.index, year))
            df_TUPLES = pd.DataFrame(index=index_product, columns=["VALUE"])
            for idx in df_TUPLES.index:
                df_TUPLES.at[idx, "VALUE"] = df_param_indexed.at[idx[0], idx[1]]
            df_TUPLES.reset_index(inplace=True)
            df_TUPLES[["SETS", "YEAR"]] = pd.DataFrame(df_TUPLES["index"].tolist(), index=df_TUPLES.index)
            df_TUPLES[sets] = pd.DataFrame(df_TUPLES["SETS"].tolist(), index=df_TUPLES.index)
            df_TUPLES = df_TUPLES.drop(["index", "SETS"], axis=1)
            df_TUPLES = df_TUPLES[sets + ["YEAR"] + ["VALUE"]]
        else:
            df_param_sub = df_param_sub.loc[~(df_param_sub == 0).all(axis=1)].reset_index()
            df_agg = df_param.drop(columns=["Parameter"] + sets).loc[~(df_param.drop(columns=["Parameter"] + sets) == 0).all(axis=1)]
            df_agg["index_col"] = df_agg.index // div
            df_agg = df_agg.groupby("index_col", as_index=False).sum()
            for y in year:
                df_param_sub[y] = df_agg[y]
            df_param_indexed = df_param_sub.set_index(sets)
            index_product = list(product(df_param_indexed.index, year))
            df_TUPLES = pd.DataFrame(index=index_product, columns=["VALUE"])
            for idx in df_TUPLES.index:
                df_TUPLES.at[idx, "VALUE"] = df_param_indexed.at[idx[0], idx[1]]
            df_TUPLES.reset_index(inplace=True)
            df_TUPLES[["SETS", "YEAR"]] = pd.DataFrame(df_TUPLES["index"].tolist(), index=df_TUPLES.index)
            df_TUPLES[sets] = pd.DataFrame(df_TUPLES["SETS"].tolist(), index=df_TUPLES.index)
            df_TUPLES = df_TUPLES.drop(["index", "SETS"], axis=1)
            df_TUPLES = df_TUPLES[sets + ["YEAR"] + ["VALUE"]]
    else:
        df_param_indexed = df_param.set_index(sets).drop(columns="Parameter")
        if df_param_indexed.empty:
            df_TUPLES = pd.DataFrame(columns=sets + ["YEAR", "VALUE"])
        else:
            index_product = list(product(df_param_indexed.index, year))
            df_TUPLES = pd.DataFrame(index=index_product, columns=["VALUE"])
            for idx in df_TUPLES.index:
                df_TUPLES.at[idx, "VALUE"] = df_param_indexed.at[idx[0], idx[1]]
            df_TUPLES.reset_index(inplace=True)
            df_TUPLES[["SETS", "YEAR"]] = pd.DataFrame(df_TUPLES["index"].tolist(), index=df_TUPLES.index)
            df_TUPLES[sets] = pd.DataFrame(df_TUPLES["SETS"].tolist(), index=df_TUPLES.index)
            df_TUPLES = df_TUPLES.drop(["index", "SETS"], axis=1)
            df_TUPLES = df_TUPLES[sets + ["YEAR"] + ["VALUE"]]
            df_TUPLES = df_TUPLES.dropna(axis=1)

    df_TUPLES.to_csv(os.path.join(path_csv, f"{param}.csv"), index=False)


def SAND_SETS_to_CSV(df, path_csv, div):
    """Deriva sets desde parametros estructurales del SAND."""
    df_param = df[df["Parameter"] == "YearSplit"].dropna(axis=1).reset_index(drop=True)
    df_param = df_param[df_param.index % div == 0]
    year = df_param.columns[df_param.columns.to_series().apply(pd.to_numeric, errors="coerce").notna()]
    df_year = pd.DataFrame(year, columns=["VALUE"])
    sets = df_param.columns[~df_param.columns.isin(year) & (df_param.columns != "Parameter")].tolist()

    for s in sets:
        df_set = pd.DataFrame(df_param[s].unique(), columns=["VALUE"])
        df_set.to_csv(os.path.join(path_csv, f"{s}.csv"), index=False)
    df_year.to_csv(os.path.join(path_csv, "YEAR.csv"), index=False)

    df_param = df[df["Parameter"] == "EmissionActivityRatio"].dropna(axis=1)
    sets = df_param.columns[~df_param.columns.isin(year) & (df_param.columns != "Parameter")].tolist()
    df_param_indexed = df_param.set_index(sets).drop(columns="Parameter").loc[
        ~(df_param.set_index(sets).drop(columns="Parameter") == 0).all(axis=1)
    ]
    if df_param_indexed.empty:
        df_TUPLES = pd.DataFrame(columns=sets + ["YEAR", "VALUE"])
    else:
        index_product = list(product(df_param_indexed.index, year))
        df_TUPLES = pd.DataFrame(index=index_product, columns=["VALUE"])
        for idx in df_TUPLES.index:
            df_TUPLES.at[idx, "VALUE"] = df_param_indexed.at[idx[0], idx[1]]
        df_TUPLES.reset_index(inplace=True)
        df_TUPLES[["SETS", "YEAR"]] = pd.DataFrame(df_TUPLES["index"].tolist(), index=df_TUPLES.index)
        df_TUPLES[sets] = pd.DataFrame(df_TUPLES["SETS"].tolist(), index=df_TUPLES.index)
        df_TUPLES = df_TUPLES.drop(["index", "SETS"], axis=1)
        df_TUPLES = df_TUPLES[sets + ["YEAR"] + ["VALUE"]]
    df_set = pd.DataFrame(df_TUPLES["EMISSION"].unique(), columns=["VALUE"])
    df_set.to_csv(os.path.join(path_csv, "EMISSION.csv"), index=False)

    df_param = df[df["Parameter"] == "OutputActivityRatio"].dropna(axis=1)
    sets = df_param.columns[~df_param.columns.isin(year) & (df_param.columns != "Parameter")].tolist()
    df_param_indexed = df_param.set_index(sets).drop(columns="Parameter").loc[
        ~(df_param.set_index(sets).drop(columns="Parameter") == 0).all(axis=1)
    ]
    if df_param_indexed.empty:
        df_TUPLES = pd.DataFrame(columns=sets + ["YEAR", "VALUE"])
    else:
        index_product = list(product(df_param_indexed.index, year))
        df_TUPLES = pd.DataFrame(index=index_product, columns=["VALUE"])
        for idx in df_TUPLES.index:
            df_TUPLES.at[idx, "VALUE"] = df_param_indexed.at[idx[0], idx[1]]
        df_TUPLES.reset_index(inplace=True)
        df_TUPLES[["SETS", "YEAR"]] = pd.DataFrame(df_TUPLES["index"].tolist(), index=df_TUPLES.index)
        df_TUPLES[sets] = pd.DataFrame(df_TUPLES["SETS"].tolist(), index=df_TUPLES.index)
        df_TUPLES = df_TUPLES.drop(["index", "SETS"], axis=1)
        df_TUPLES = df_TUPLES[sets + ["YEAR"] + ["VALUE"]]
    for s in sets:
        df_set = pd.DataFrame(df_TUPLES[s].unique(), columns=["VALUE"])
        df_set.to_csv(os.path.join(path_csv, f"{s}.csv"), index=False)

    df_param = df[df["Parameter"] == "CapacityToActivityUnit"].dropna(axis=1)
    sets = df_param.columns[~df_param.columns.isin(year) & (df_param.columns != "Parameter")].tolist()
    sets.remove("Time indipendent variables")
    df_TUPLES = df_param.drop(["Parameter"], axis=1).rename(columns={"Time indipendent variables": "VALUE"}).loc[
        ~(df_param.drop(["Parameter"], axis=1).rename(columns={"Time indipendent variables": "VALUE"})["VALUE"] == 0)
    ]
    for s in sets:
        df_set = pd.DataFrame(df_TUPLES[s].unique(), columns=["VALUE"])
        df_set.to_csv(os.path.join(path_csv, f"{s}.csv"), index=False)

print("SAND_to_CSV y SAND_SETS_to_CSV definidos.")

SAND_to_CSV y SAND_SETS_to_CSV definidos.


In [3]:
def completar_Matrix_Act_Ratio(path_csv, variable):
    df = pd.read_csv(path_csv + variable)
    regions = df["REGION"].unique()
    technologies = pd.read_csv(path_csv + "TECHNOLOGY.csv", dtype=str)["VALUE"].unique()
    fuels = pd.read_csv(path_csv + "FUEL.csv", dtype=str)["VALUE"].unique()
    modes = pd.read_csv(path_csv + "MODE_OF_OPERATION.csv")["VALUE"].unique()
    years = pd.read_csv(path_csv + "YEAR.csv")["VALUE"].unique()

    all_combinations = pd.DataFrame(itertools.product(regions, technologies, fuels, modes, years),
        columns=["REGION", "TECHNOLOGY", "FUEL", "MODE_OF_OPERATION", "YEAR"])
    result = all_combinations.merge(df, on=["REGION", "TECHNOLOGY", "MODE_OF_OPERATION", "FUEL", "YEAR"], how="left")
    result.dropna(subset=["VALUE"], inplace=True)
    result.to_csv(path_csv + variable, index=False)


def completar_Matrix_Emission(path_csv, variable):
    df = pd.read_csv(path_csv + variable)
    regions = df["REGION"].unique()
    technologies = pd.read_csv(path_csv + "TECHNOLOGY.csv")["VALUE"].unique()
    emission = pd.read_csv(path_csv + "EMISSION.csv")["VALUE"].unique()
    modes = pd.read_csv(path_csv + "MODE_OF_OPERATION.csv")["VALUE"].unique()
    years = pd.read_csv(path_csv + "YEAR.csv")["VALUE"].unique()

    all_combinations = pd.DataFrame(itertools.product(regions, technologies, emission, modes, years),
        columns=["REGION", "TECHNOLOGY", "EMISSION", "MODE_OF_OPERATION", "YEAR"])
    result = all_combinations.merge(df, on=["REGION", "TECHNOLOGY", "EMISSION", "MODE_OF_OPERATION", "YEAR"], how="left")
    result.dropna(subset=["VALUE"], inplace=True)
    result.to_csv(path_csv + variable, index=False)


def completar_Matrix_Cost(path_csv, variable):
    df = pd.read_csv(path_csv + variable)
    regions = df["REGION"].unique()
    technologies = pd.read_csv(path_csv + "TECHNOLOGY.csv")["VALUE"].unique()
    modes = pd.read_csv(path_csv + "MODE_OF_OPERATION.csv")["VALUE"].unique()
    years = pd.read_csv(path_csv + "YEAR.csv")["VALUE"].unique()

    all_combinations = pd.DataFrame(itertools.product(regions, technologies, modes, years),
        columns=["REGION", "TECHNOLOGY", "MODE_OF_OPERATION", "YEAR"])
    result = all_combinations.merge(df, on=["REGION", "TECHNOLOGY", "MODE_OF_OPERATION", "YEAR"], how="left")
    result.dropna(subset=["VALUE"], inplace=True)
    result.to_csv(path_csv + variable, index=False)


def process_and_save_emission_ratios(emission_activity_path, input_activity_path, output_path, path_csv):
    df_emission = pd.read_csv(path_csv + emission_activity_path)
    df_input = pd.read_csv(path_csv + input_activity_path)

    keys = ["REGION", "TECHNOLOGY", "MODE_OF_OPERATION", "YEAR"]
    for d in (df_emission, df_input):
        for k in keys:
            if k in d.columns:
                d[k] = d[k].astype(str).str.strip()

    merged = pd.merge(df_emission, df_input, on=keys, how="left").query("VALUE_x != 0 and VALUE_y != 0").assign(
        VALUE=lambda x: x["VALUE_x"] * x["VALUE_y"]
    )
    drop_cols = [c for c in ["VALUE_x", "VALUE_y", "FUEL"] if c in merged.columns]
    merged = merged.drop(columns=drop_cols)
    merged_unique = merged.groupby(["REGION", "TECHNOLOGY", "EMISSION", "MODE_OF_OPERATION", "YEAR"], as_index=False).agg({"VALUE": "first"})

    final_merged = pd.merge(df_emission, merged_unique,
        on=["REGION", "TECHNOLOGY", "EMISSION", "MODE_OF_OPERATION", "YEAR"], how="left", suffixes=("_df1", "_df4")
    ).assign(
        VALUE=lambda x: x.apply(
            lambda row: row["VALUE_df4"] if pd.notnull(row["VALUE_df4"]) and row["VALUE_df1"] != row["VALUE_df4"] else row["VALUE_df1"],
            axis=1,
        )
    ).loc[:, ["REGION", "TECHNOLOGY", "EMISSION", "MODE_OF_OPERATION", "YEAR", "VALUE"]]

    final_merged.to_csv(path_csv + output_path, index=False)


def filter_params_by_sets(path_csv, df_colombia):
    parameter_list = df_colombia["Parameter"].unique()
    for p in parameter_list:
        fpath = path_csv + p + ".csv"
        if not os.path.exists(fpath):
            continue
        df_p = pd.read_csv(fpath)
        sets_cols = df_p.columns.drop("VALUE")
        if "REGION2" in sets_cols:
            sets_cols = df_p.columns.drop(["REGION2", "VALUE"])
        for s in sets_cols:
            set_fpath = path_csv + s + ".csv"
            if os.path.exists(set_fpath):
                df_sets = pd.read_csv(set_fpath)
                df_p = df_p[df_p[s].isin(df_sets.VALUE.tolist())]
        df_p.to_csv(fpath, index=False)

print("Helpers de matrices y filtrado definidos.")

Helpers de matrices y filtrado definidos.


In [4]:
PARAM_INDEX: dict[str, list[str]] = {
    "YearSplit": ["TIMESLICE", "YEAR"],
    "DiscountRate": ["REGION"],
    "DepreciationMethod": ["REGION"],
    "OperationalLife": ["REGION", "TECHNOLOGY"],
    "CapacityToActivityUnit": ["REGION", "TECHNOLOGY"],
    "CapacityOfOneTechnologyUnit": ["REGION", "TECHNOLOGY", "YEAR"],
    "CapacityFactor": ["REGION", "TECHNOLOGY", "TIMESLICE", "YEAR"],
    "AvailabilityFactor": ["REGION", "TECHNOLOGY", "YEAR"],
    "ResidualCapacity": ["REGION", "TECHNOLOGY", "YEAR"],
    "InputActivityRatio": ["REGION", "TECHNOLOGY", "FUEL", "MODE_OF_OPERATION", "YEAR"],
    "OutputActivityRatio": ["REGION", "TECHNOLOGY", "FUEL", "MODE_OF_OPERATION", "YEAR"],
    "CapitalCost": ["REGION", "TECHNOLOGY", "YEAR"],
    "VariableCost": ["REGION", "TECHNOLOGY", "MODE_OF_OPERATION", "YEAR"],
    "FixedCost": ["REGION", "TECHNOLOGY", "YEAR"],
    "TotalAnnualMaxCapacity": ["REGION", "TECHNOLOGY", "YEAR"],
    "TotalAnnualMinCapacity": ["REGION", "TECHNOLOGY", "YEAR"],
    "TotalAnnualMaxCapacityInvestment": ["REGION", "TECHNOLOGY", "YEAR"],
    "TotalAnnualMinCapacityInvestment": ["REGION", "TECHNOLOGY", "YEAR"],
    "TotalTechnologyAnnualActivityUpperLimit": ["REGION", "TECHNOLOGY", "YEAR"],
    "TotalTechnologyAnnualActivityLowerLimit": ["REGION", "TECHNOLOGY", "YEAR"],
    "TotalTechnologyModelPeriodActivityUpperLimit": ["REGION", "TECHNOLOGY"],
    "TotalTechnologyModelPeriodActivityLowerLimit": ["REGION", "TECHNOLOGY"],
    "ReserveMarginTagTechnology": ["REGION", "TECHNOLOGY", "YEAR"],
    "ReserveMarginTagFuel": ["REGION", "FUEL", "YEAR"],
    "ReserveMargin": ["REGION", "YEAR"],
    "RETagTechnology": ["REGION", "TECHNOLOGY", "YEAR"],
    "RETagFuel": ["REGION", "FUEL", "YEAR"],
    "REMinProductionTarget": ["REGION", "YEAR"],
    "AccumulatedAnnualDemand": ["REGION", "FUEL", "YEAR"],
    "SpecifiedAnnualDemand": ["REGION", "FUEL", "YEAR"],
    "SpecifiedDemandProfile": ["REGION", "FUEL", "TIMESLICE", "YEAR"],
    "EmissionActivityRatio": ["REGION", "TECHNOLOGY", "EMISSION", "MODE_OF_OPERATION", "YEAR"],
    "EmissionsPenalty": ["REGION", "EMISSION", "YEAR"],
    "AnnualExogenousEmission": ["REGION", "EMISSION", "YEAR"],
    "AnnualEmissionLimit": ["REGION", "EMISSION", "YEAR"],
    "ModelPeriodExogenousEmission": ["REGION", "EMISSION"],
    "ModelPeriodEmissionLimit": ["REGION", "EMISSION"],
    "DiscountRateIdv": ["REGION", "TECHNOLOGY"],
}


def normalize_mode_of_operation_scalar(v):
    if v is None:
        return ""
    try:
        if pd.isna(v):
            return ""
    except (TypeError, ValueError):
        pass
    if isinstance(v, (int, float)):
        if isinstance(v, float) and not float(v).is_integer():
            return str(v).strip()
        return str(int(v))
    s = str(v).strip()
    if s == "" or s.lower() == "nan":
        return ""
    try:
        f = float(s)
        if f.is_integer():
            return str(int(f))
        return str(f)
    except ValueError:
        return s


def normalize_mode_of_operation_series(series):
    return series.map(normalize_mode_of_operation_scalar)


def normalize_mode_of_operation_in_csv_dir(csv_dir):
    moo_path = os.path.join(csv_dir, "MODE_OF_OPERATION.csv")
    if os.path.exists(moo_path):
        df = pd.read_csv(moo_path)
        if not df.empty and "VALUE" in df.columns:
            df["VALUE"] = normalize_mode_of_operation_series(df["VALUE"])
            df = df[df["VALUE"] != ""]
            df = df.drop_duplicates(subset=["VALUE"], keep="first")
            df.to_csv(moo_path, index=False)
    for pname, cols in PARAM_INDEX.items():
        if "MODE_OF_OPERATION" not in cols:
            continue
        ppath = os.path.join(csv_dir, f"{pname}.csv")
        if not os.path.exists(ppath):
            continue
        df = pd.read_csv(ppath)
        if df.empty or "MODE_OF_OPERATION" not in df.columns:
            continue
        df["MODE_OF_OPERATION"] = normalize_mode_of_operation_series(df["MODE_OF_OPERATION"])
        df.to_csv(ppath, index=False)


def eliminar_valores_fuera_de_indices(csv_dir):
    param_files = [f for f in os.listdir(csv_dir) if f.endswith(".csv") and f.replace(".csv", "") in PARAM_INDEX]
    for pfile in param_files:
        path = os.path.join(csv_dir, pfile)
        df = pd.read_csv(path)
        if df.empty:
            continue
        cols = [c for c in df.columns if c != "VALUE"]
        modified = False
        for col in cols:
            set_path = os.path.join(csv_dir, f"{col}.csv")
            if os.path.exists(set_path):
                set_df = pd.read_csv(set_path)
                if "VALUE" in set_df.columns:
                    valid = set_df["VALUE"].tolist()
                    before = len(df)
                    df = df[df[col].isin(valid)]
                    if len(df) < before:
                        modified = True
        if modified:
            df.to_csv(path, index=False)


def reorder_activity_ratio_csvs_for_dataportal(csv_dir):
    for fname in ("InputActivityRatio.csv", "OutputActivityRatio.csv"):
        fpath = os.path.join(csv_dir, fname)
        if os.path.exists(fpath):
            df = pd.read_csv(fpath)
            expected = ["REGION", "TECHNOLOGY", "FUEL", "MODE_OF_OPERATION", "YEAR", "VALUE"]
            if all(c in df.columns for c in expected):
                df["MODE_OF_OPERATION"] = normalize_mode_of_operation_series(df["MODE_OF_OPERATION"])
                df[expected].to_csv(fpath, index=False)

print("Helpers de post-procesado definidos.")

Helpers de post-procesado definidos.


## 3. Generar CSVs (pipeline completo)

Si la fuente es Excel:

1. Lee `Parameters` del Excel.
2. `SAND_SETS_to_CSV` y luego `SAND_to_CSV` para cada parámetro único.
3. `filter_params_by_sets`.
4. `completar_Matrix_*` (Act_Ratio ×2, Emission, Cost).
5. `process_and_save_emission_ratios`.
6. Post: `normalize_mode_of_operation`, `eliminar_valores_fuera_de_indices`, `completar_Matrix_*` otra vez (idempotente), reorder.

Si la fuente es `CSV_DIR_IN`, `csv_dir` apunta directo a la carpeta y se omite la generación.

In [5]:
import time, shutil

if EXCEL_PATH is not None:
    assert EXCEL_PATH.is_file(), f"No existe el Excel: {EXCEL_PATH}"

    if CSV_DIR_OUT.exists():
        shutil.rmtree(CSV_DIR_OUT)
    CSV_DIR_OUT.mkdir(parents=True, exist_ok=True)

    path_csv = str(CSV_DIR_OUT) + os.sep
    div = 1   # 96/div = 96 (sin submuestreo de timeslices)

    t0 = time.perf_counter()
    print(f"  Leyendo Excel: {EXCEL_PATH}")
    df = pd.read_excel(EXCEL_PATH, sheet_name="Parameters")
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].str.strip()
    print(f'  Parametros encontrados: {df["Parameter"].nunique()}')

    print("  Generando sets...")
    SAND_SETS_to_CSV(df, path_csv, 96 / div)

    print("  Generando parametros...")
    for p in df["Parameter"].dropna().unique():
        SAND_to_CSV(df, p, path_csv, 96 / div)

    print("  Filtrando por sets...")
    filter_params_by_sets(path_csv, df)

    print("  Completando matrices...")
    completar_Matrix_Act_Ratio(path_csv, "InputActivityRatio.csv")
    completar_Matrix_Act_Ratio(path_csv, "OutputActivityRatio.csv")
    if os.path.exists(path_csv + "EMISSION.csv"):
        completar_Matrix_Emission(path_csv, "EmissionActivityRatio.csv")
    completar_Matrix_Cost(path_csv, "VariableCost.csv")

    if os.path.exists(path_csv + "EMISSION.csv"):
        print("  Procesando emisiones a la entrada...")
        process_and_save_emission_ratios(
            "EmissionActivityRatio.csv", "InputActivityRatio.csv",
            "EmissionActivityRatio.csv", path_csv,
        )

    print("  Post-procesado...")
    normalize_mode_of_operation_in_csv_dir(str(CSV_DIR_OUT))
    eliminar_valores_fuera_de_indices(str(CSV_DIR_OUT))
    completar_Matrix_Act_Ratio(path_csv, "InputActivityRatio.csv")
    completar_Matrix_Act_Ratio(path_csv, "OutputActivityRatio.csv")
    if os.path.exists(path_csv + "EMISSION.csv"):
        completar_Matrix_Emission(path_csv, "EmissionActivityRatio.csv")
    completar_Matrix_Cost(path_csv, "VariableCost.csv")
    if os.path.exists(path_csv + "EMISSION.csv"):
        process_and_save_emission_ratios(
            "EmissionActivityRatio.csv", "InputActivityRatio.csv",
            "EmissionActivityRatio.csv", path_csv,
        )
    reorder_activity_ratio_csvs_for_dataportal(str(CSV_DIR_OUT))

    csv_dir = CSV_DIR_OUT
    elapsed = time.perf_counter() - t0
    n_csvs = len(list(CSV_DIR_OUT.glob("*.csv")))
    print(f"\nGeneracion completada en {elapsed:.2f} s -> {n_csvs} CSVs en {csv_dir}")
else:
    assert CSV_DIR_IN is not None and CSV_DIR_IN.is_dir(), f"No existe la carpeta CSV: {CSV_DIR_IN}"
    csv_dir = CSV_DIR_IN
    n_csvs = len(list(csv_dir.glob("*.csv")))
    print(f"Usando CSVs existentes: {csv_dir} ({n_csvs} CSVs)")

  Leyendo Excel: /Users/davidbedoya0/Downloads/29_04_26_SAND_INTEGRADO_CN_V2 _RM__Parameters_SAND.xlsx


/var/folders/j4/sw28wy4s2q5gxt0vkbtlwt1h0000gn/T/ipykernel_44621/3694557828.py:16: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


  Parametros encontrados: 37
  Generando sets...


  Generando parametros...


  Filtrando por sets...


  Completando matrices...


  Procesando emisiones a la entrada...


  Post-procesado...



Generacion completada en 27.31 s -> 44 CSVs en /Users/davidbedoya0/Documents/UPME/Demanda/Repositorio/OSEMOSYS_FINAL_2/Osemosys_UPME/backend/app/simulation/tmp_osemosys_csvs


## 4. AbstractModel OSeMOSYS (inline completo)

Definición completa del modelo OSeMOSYS — sets, parámetros, variables, función objetivo y todas las restricciones — embebida en el notebook.

Este código se reproduce verbatim desde `app/simulation/core/model_definition.py` del proyecto. No hay imports a `app.*`.

In [6]:
"""Definición completa del AbstractModel OSeMOSYS.

Replica fielmente la celda 3 del notebook osemosys_notebook_UPME_OPT_01.ipynb:
sets, parámetros, variables, función objetivo y TODAS las restricciones
en un solo archivo.

OSeMOSYS: Open Source energy MOdeling SYStem
Copyright [2010-2015] [OSeMOSYS Forum steering committee see: www.osemosys.org]
Licensed under the Apache License, Version 2.0

Estructura del archivo:
  - Sets: YEAR, TECHNOLOGY, TIMESLICE, FUEL, EMISSION, MODE_OF_OPERATION, REGION; opcionales STORAGE, SEASON, DAYTYPE, DAILYTIMEBRACKET, UDC.
  - Parameters: demanda, rendimientos (CapacityFactor, ActivityRatios), costes, límites de capacidad/actividad, emisiones, UDC, almacenamiento.
  - Variables: NewCapacity, RateOfActivity, costes descontados, emisiones, reserva, RE, almacenamiento.
  - Objective: minimizar suma de TotalDiscountedCost por región y año.
  - Constraints: capacidad, balance energético, costes, límites, emisiones, reserve margin, UDC, almacenamiento.
"""

from __future__ import annotations

from pyomo.environ import (
    AbstractModel,
    Constraint,
    NonNegativeIntegers,
    NonNegativeReals,
    Objective,
    Param,
    Reals,
    Set,
    Var,
    minimize,
)


def create_abstract_model(
    has_storage: bool = False,
    has_udc: bool = True,
) -> AbstractModel:
    """Construye el AbstractModel OSeMOSYS completo.

    Parameters
    ----------
    has_storage : bool
        Si True, incluye sets/params/vars/constraints de almacenamiento.
    has_udc : bool
        Si True, incluye User-Defined Constraints.

    Returns
    -------
    AbstractModel listo para ``model.create_instance(data)``.
    """
    model = AbstractModel()

    # ====================================================================
    #    Sets (conjuntos que indexan parámetros y variables)
    # ====================================================================

    model.YEAR = Set(ordered=True)
    model.TECHNOLOGY = Set()
    model.TIMESLICE = Set(ordered=True)
    model.FUEL = Set()
    model.EMISSION = Set()
    model.MODE_OF_OPERATION = Set()
    model.REGION = Set()

    if has_storage:
        model.STORAGE = Set()
        model.SEASON = Set()
        model.DAYTYPE = Set()
        model.DAILYTIMEBRACKET = Set()
        model.STORAGEINTRADAY = Set()
        model.STORAGEINTRAYEAR = Set()

    model.FLEXIBLEDEMANDTYPE = Set()
    model.UDC = Set()

    # ====================================================================
    #    Parameters — Global (YearSplit, descuento, vida operativa, factores de recuperación)
    # ====================================================================

    model.YearSplit = Param(model.TIMESLICE, model.YEAR)
    model.DiscountRate = Param(model.REGION, default=0.05)

    def DiscountRateIdv_init(m, r, t):
        return m.DiscountRate[r]
    model.DiscountRateIdv = Param(
        model.REGION, model.TECHNOLOGY,
        within=NonNegativeReals, initialize=DiscountRateIdv_init, mutable=True,
    )
    model.OperationalLife = Param(model.REGION, model.TECHNOLOGY, default=1)

    def CapitalRecoveryFactor_rule(m, r, t):
        dr = m.DiscountRateIdv[r, t]
        ol = m.OperationalLife[r, t]
        return (1 - (1 + dr)**(-1)) / (1 - (1 + dr)**(-ol))
    model.CapitalRecoveryFactor = Param(
        model.REGION, model.TECHNOLOGY,
        initialize=CapitalRecoveryFactor_rule, within=Reals, mutable=True,
    )

    def PvAnnuity_rule(m, r, t):
        dr = m.DiscountRate[r]
        ol = m.OperationalLife[r, t]
        return (1 - (1 + dr)**(-ol)) * (1 + dr) / dr
    model.PvAnnuity = Param(
        model.REGION, model.TECHNOLOGY,
        initialize=PvAnnuity_rule, within=Reals, mutable=True,
    )

    model.DepreciationMethod = Param(model.REGION, default=1)

    # ====================================================================
    #    Parameters — Demands
    # ====================================================================

    model.AccumulatedAnnualDemand = Param(model.REGION, model.FUEL, model.YEAR, default=0)
    model.SpecifiedAnnualDemand = Param(model.REGION, model.FUEL, model.YEAR, default=0)
    model.SpecifiedDemandProfile = Param(
        model.REGION, model.FUEL, model.TIMESLICE, model.YEAR, default=0,
    )

    def Demand_init(m, r, l, f, y):
        if m.SpecifiedAnnualDemand[r, f, y] > 0:
            return m.SpecifiedAnnualDemand[r, f, y] * m.SpecifiedDemandProfile[r, f, l, y]
        return 0.0
    model.Demand = Param(
        model.REGION, model.TIMESLICE, model.FUEL, model.YEAR,
        initialize=Demand_init, default=0.0,
    )

    # ====================================================================
    #    Parameters — Performance
    # ====================================================================

    model.CapacityToActivityUnit = Param(model.REGION, model.TECHNOLOGY, default=1)
    model.CapacityFactor = Param(
        model.REGION, model.TECHNOLOGY, model.TIMESLICE, model.YEAR, default=1,
    )
    model.AvailabilityFactor = Param(model.REGION, model.TECHNOLOGY, model.YEAR, default=1)
    model.ResidualCapacity = Param(model.REGION, model.TECHNOLOGY, model.YEAR, default=0)
    model.InputActivityRatio = Param(
        model.REGION, model.TECHNOLOGY, model.FUEL, model.MODE_OF_OPERATION, model.YEAR,
        default=0,
    )
    model.OutputActivityRatio = Param(
        model.REGION, model.TECHNOLOGY, model.FUEL, model.MODE_OF_OPERATION, model.YEAR,
        default=0,
    )

    # ====================================================================
    #    Parameters — Technology Costs
    # ====================================================================

    model.CapitalCost = Param(model.REGION, model.TECHNOLOGY, model.YEAR, default=0.000001)
    model.VariableCost = Param(
        model.REGION, model.TECHNOLOGY, model.MODE_OF_OPERATION, model.YEAR,
        default=0.000001,
    )
    model.FixedCost = Param(model.REGION, model.TECHNOLOGY, model.YEAR, default=0)

    # ====================================================================
    #    Parameters — Capacity Constraints
    # ====================================================================

    model.CapacityOfOneTechnologyUnit = Param(
        model.REGION, model.TECHNOLOGY, model.YEAR, default=0,
    )
    model.TotalAnnualMaxCapacity = Param(
        model.REGION, model.TECHNOLOGY, model.YEAR, default=9999999,
    )
    model.TotalAnnualMinCapacity = Param(
        model.REGION, model.TECHNOLOGY, model.YEAR, default=0,
    )

    # ====================================================================
    #    Parameters — Investment Constraints
    # ====================================================================

    model.TotalAnnualMaxCapacityInvestment = Param(
        model.REGION, model.TECHNOLOGY, model.YEAR, default=9999999,
    )
    model.TotalAnnualMinCapacityInvestment = Param(
        model.REGION, model.TECHNOLOGY, model.YEAR, default=0,
    )

    # ====================================================================
    #    Parameters — Activity Constraints
    # ====================================================================

    model.TotalTechnologyAnnualActivityUpperLimit = Param(
        model.REGION, model.TECHNOLOGY, model.YEAR, default=9999999,
    )
    model.TotalTechnologyAnnualActivityLowerLimit = Param(
        model.REGION, model.TECHNOLOGY, model.YEAR, default=0,
    )
    model.TotalTechnologyModelPeriodActivityUpperLimit = Param(
        model.REGION, model.TECHNOLOGY, default=9999999,
    )
    model.TotalTechnologyModelPeriodActivityLowerLimit = Param(
        model.REGION, model.TECHNOLOGY, default=0,
    )

    # ====================================================================
    #    Parameters — Reserve Margin
    # ====================================================================

    model.ReserveMarginTagTechnology = Param(
        model.REGION, model.TECHNOLOGY, model.YEAR, default=0,
    )
    model.ReserveMarginTagFuel = Param(model.REGION, model.FUEL, model.YEAR, default=0)
    model.ReserveMargin = Param(model.REGION, model.YEAR, default=1)

    # ====================================================================
    #    Parameters — RE Generation Target
    # ====================================================================

    model.RETagTechnology = Param(model.REGION, model.TECHNOLOGY, model.YEAR, default=0)
    model.RETagFuel = Param(model.REGION, model.FUEL, model.YEAR, default=0)
    model.REMinProductionTarget = Param(model.REGION, model.YEAR, default=0)

    # ====================================================================
    #    Parameters — Emissions & Penalties
    # ====================================================================

    model.EmissionActivityRatio = Param(
        model.REGION, model.TECHNOLOGY, model.EMISSION, model.MODE_OF_OPERATION, model.YEAR,
        default=0,
    )
    model.EmissionsPenalty = Param(model.REGION, model.EMISSION, model.YEAR, default=0)
    model.AnnualExogenousEmission = Param(
        model.REGION, model.EMISSION, model.YEAR, default=0,
    )
    model.AnnualEmissionLimit = Param(
        model.REGION, model.EMISSION, model.YEAR, default=9999999,
    )
    model.ModelPeriodExogenousEmission = Param(model.REGION, model.EMISSION, default=0)
    model.ModelPeriodEmissionLimit = Param(model.REGION, model.EMISSION, default=9999999)

    # ====================================================================
    #    Parameters — MUIO
    # ====================================================================

    model.InputToNewCapacityRatio = Param(
        model.REGION, model.TECHNOLOGY, model.FUEL, model.YEAR, within=Reals, default=0,
    )
    model.InputToTotalCapacityRatio = Param(
        model.REGION, model.TECHNOLOGY, model.FUEL, model.YEAR, within=Reals, default=0,
    )
    model.TechnologyActivityByModeLowerLimit = Param(
        model.REGION, model.TECHNOLOGY, model.MODE_OF_OPERATION, model.YEAR,
        within=Reals, default=0,
    )
    model.TechnologyActivityByModeUpperLimit = Param(
        model.REGION, model.TECHNOLOGY, model.MODE_OF_OPERATION, model.YEAR,
        within=Reals, default=0,
    )
    model.TechnologyActivityDecreaseByModeLimit = Param(
        model.REGION, model.TECHNOLOGY, model.MODE_OF_OPERATION, model.YEAR,
        within=Reals, default=0,
    )
    model.TechnologyActivityIncreaseByModeLimit = Param(
        model.REGION, model.TECHNOLOGY, model.MODE_OF_OPERATION, model.YEAR,
        within=Reals, default=0,
    )
    model.EmissionToActivityChangeRatio = Param(
        model.REGION, model.TECHNOLOGY, model.EMISSION, model.MODE_OF_OPERATION, model.YEAR,
        within=Reals, default=0,
    )

    # ====================================================================
    #    Parameters — UDC (User-Defined Constraints)
    # ====================================================================

    if has_udc:
        model.UDCMultiplierTotalCapacity = Param(
            model.REGION, model.TECHNOLOGY, model.UDC, model.YEAR,
            within=Reals, default=0,
        )
        model.UDCMultiplierNewCapacity = Param(
            model.REGION, model.TECHNOLOGY, model.UDC, model.YEAR,
            within=Reals, default=0,
        )
        model.UDCMultiplierActivity = Param(
            model.REGION, model.TECHNOLOGY, model.UDC, model.YEAR,
            within=Reals, default=0,
        )
        model.UDCConstant = Param(
            model.REGION, model.UDC, model.YEAR, within=Reals, default=0,
        )
        model.UDCTag = Param(model.REGION, model.UDC, within=Reals, default=2)

    # ====================================================================
    #    Parameters — Storage
    # ====================================================================

    if has_storage:
        model.DaySplit = Param(model.DAILYTIMEBRACKET, model.YEAR, default=0.00137)
        model.Conversionls = Param(model.TIMESLICE, model.SEASON, default=0)
        model.Conversionld = Param(model.TIMESLICE, model.DAYTYPE, default=0)
        model.Conversionlh = Param(model.TIMESLICE, model.DAILYTIMEBRACKET, default=0)
        model.DaysInDayType = Param(model.SEASON, model.DAYTYPE, model.YEAR, default=7)
        model.TechnologyToStorage = Param(
            model.REGION, model.TECHNOLOGY, model.STORAGE, model.MODE_OF_OPERATION, default=0,
        )
        model.TechnologyFromStorage = Param(
            model.REGION, model.TECHNOLOGY, model.STORAGE, model.MODE_OF_OPERATION, default=0,
        )
        model.StorageLevelStart = Param(model.REGION, model.STORAGE, default=0.0000001)
        model.StorageMaxChargeRate = Param(model.REGION, model.STORAGE, default=9999999)
        model.StorageMaxDischargeRate = Param(model.REGION, model.STORAGE, default=9999999)
        model.MinStorageCharge = Param(model.REGION, model.STORAGE, model.YEAR, default=0)
        model.OperationalLifeStorage = Param(model.REGION, model.STORAGE, default=0)
        model.CapitalCostStorage = Param(
            model.REGION, model.STORAGE, model.YEAR, default=0,
        )
        model.ResidualStorageCapacity = Param(
            model.REGION, model.STORAGE, model.YEAR, default=0,
        )

    # ====================================================================
    #    Parameters — Disposal / Recovery (Max B/C)
    # ====================================================================

    model.DisposalCostPerCapacity = Param(model.REGION, model.TECHNOLOGY, default=0.0)
    model.RecoveryValuePerCapacity = Param(model.REGION, model.TECHNOLOGY, default=0.0)

    # ====================================================================
    #    Variables — Capacity
    # ====================================================================

    model.NumberOfNewTechnologyUnits = Var(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        domain=NonNegativeIntegers, initialize=0,
    )
    model.NewCapacity = Var(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )

    # ====================================================================
    #    Variables — Activity
    # ====================================================================

    model.RateOfActivity = Var(
        model.REGION, model.TIMESLICE, model.TECHNOLOGY, model.MODE_OF_OPERATION, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )

    # ====================================================================
    #    Variables — Costing
    # ====================================================================

    model.VariableOperatingCost = Var(
        model.REGION, model.TECHNOLOGY, model.TIMESLICE, model.YEAR,
        domain=Reals, initialize=0.0,
    )
    model.SalvageValue = Var(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )
    model.DiscountedSalvageValue = Var(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )
    model.OperatingCost = Var(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        domain=Reals, initialize=0.0,
    )
    model.CapitalInvestment = Var(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )
    model.DiscountedCapitalInvestment = Var(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )
    model.DiscountedOperatingCost = Var(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        domain=Reals, initialize=0.0,
    )
    model.AnnualVariableOperatingCost = Var(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        domain=Reals, initialize=0.0,
    )
    model.AnnualFixedOperatingCost = Var(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )
    model.TotalDiscountedCostByTechnology = Var(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        domain=Reals, initialize=0.0,
    )
    model.TotalDiscountedCost = Var(
        model.REGION, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )

    # ====================================================================
    #    Variables — Reserve Margin
    # ====================================================================

    model.TotalCapacityInReserveMargin = Var(
        model.REGION, model.YEAR, domain=NonNegativeReals, initialize=0.0,
    )
    model.DemandNeedingReserveMargin = Var(
        model.REGION, model.TIMESLICE, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )

    # ====================================================================
    #    Variables — RE Gen Target
    # ====================================================================

    model.TotalREProductionAnnual = Var(model.REGION, model.YEAR, initialize=0.0)
    model.RETotalProductionOfTargetFuelAnnual = Var(
        model.REGION, model.YEAR, initialize=0.0,
    )
    model.TotalTechnologyModelPeriodActivity = Var(
        model.REGION, model.TECHNOLOGY, initialize=0.0,
    )

    # ====================================================================
    #    Variables — Emissions
    # ====================================================================

    model.AnnualTechnologyEmissionByMode = Var(
        model.REGION, model.TECHNOLOGY, model.EMISSION, model.MODE_OF_OPERATION, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )
    model.AnnualTechnologyEmission = Var(
        model.REGION, model.TECHNOLOGY, model.EMISSION, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )
    model.AnnualTechnologyEmissionPenaltyByEmission = Var(
        model.REGION, model.TECHNOLOGY, model.EMISSION, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )
    model.AnnualTechnologyEmissionsPenalty = Var(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )
    model.DiscountedTechnologyEmissionsPenalty = Var(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )
    model.AnnualEmissions = Var(
        model.REGION, model.EMISSION, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )
    model.ModelPeriodEmissions = Var(
        model.REGION, model.EMISSION,
        domain=NonNegativeReals, initialize=0.0,
    )

    # ====================================================================
    #    Variables — Storage
    # ====================================================================

    if has_storage:
        model.NewStorageCapacity = Var(
            model.REGION, model.STORAGE, model.YEAR,
            domain=NonNegativeReals, initialize=0.0,
        )
        model.SalvageValueStorage = Var(
            model.REGION, model.STORAGE, model.YEAR,
            domain=NonNegativeReals, initialize=0.0,
        )
        model.StorageLevelYearStart = Var(
            model.REGION, model.STORAGE, model.YEAR,
            domain=NonNegativeReals, initialize=0.0,
        )
        model.StorageLevelYearFinish = Var(
            model.REGION, model.STORAGE, model.YEAR,
            domain=NonNegativeReals, initialize=0.0,
        )
        model.RateOfStorageCharge = Var(
            model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
            model.DAILYTIMEBRACKET, model.YEAR,
            domain=NonNegativeReals, initialize=0.0,
        )
        model.RateOfStorageDischarge = Var(
            model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
            model.DAILYTIMEBRACKET, model.YEAR,
            domain=NonNegativeReals, initialize=0.0,
        )
        model.NetChargeWithinYear = Var(
            model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
            model.DAILYTIMEBRACKET, model.YEAR,
            domain=Reals, initialize=0.0,
        )
        model.NetChargeWithinDay = Var(
            model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
            model.DAILYTIMEBRACKET, model.YEAR,
            domain=Reals, initialize=0.0,
        )
        model.StorageLevelSeasonStart = Var(
            model.REGION, model.STORAGE, model.SEASON, model.YEAR,
            domain=NonNegativeReals, initialize=0.0,
        )
        model.StorageLevelDayTypeStart = Var(
            model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE, model.YEAR,
            domain=NonNegativeReals, initialize=0.0,
        )
        model.StorageLevelDayTypeFinish = Var(
            model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE, model.YEAR,
            domain=NonNegativeReals, initialize=0.0,
        )
        model.StorageLowerLimit = Var(
            model.REGION, model.STORAGE, model.YEAR,
            domain=NonNegativeReals, initialize=0.0,
        )
        model.StorageUpperLimit = Var(
            model.REGION, model.STORAGE, model.YEAR,
            domain=NonNegativeReals, initialize=0.0,
        )
        model.AccumulatedNewStorageCapacity = Var(
            model.REGION, model.STORAGE, model.YEAR,
            domain=NonNegativeReals, initialize=0.0,
        )
        model.CapitalInvestmentStorage = Var(
            model.REGION, model.STORAGE, model.YEAR,
            domain=NonNegativeReals, initialize=0.0,
        )
        model.DiscountedCapitalInvestmentStorage = Var(
            model.REGION, model.STORAGE, model.YEAR,
            domain=NonNegativeReals, initialize=0.0,
        )
        model.DiscountedSalvageValueStorage = Var(
            model.REGION, model.STORAGE, model.YEAR,
            domain=NonNegativeReals, initialize=0.0,
        )
        model.TotalDiscountedStorageCost = Var(
            model.REGION, model.STORAGE, model.YEAR,
            domain=NonNegativeReals, initialize=0.0,
        )

    # ====================================================================
    #    Variables — Disposal / Recovery
    # ====================================================================

    model.DisposalCost = Var(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )
    model.DiscountedDisposalCost = Var(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )
    model.RecoveryValue = Var(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )
    model.DiscountedRecoveryValue = Var(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        domain=NonNegativeReals, initialize=0.0,
    )

    # ####################################################################
    #    Objective Function
    # ####################################################################

    def ObjectiveFunction_rule(m):
        return sum(m.TotalDiscountedCost[r, y] for r in m.REGION for y in m.YEAR)
    model.OBJ = Objective(rule=ObjectiveFunction_rule, sense=minimize)

    # ####################################################################
    #    Constraints — Capacity Adequacy A
    # ####################################################################

    def TotalNewCapacity_2_rule(m, r, t, y):
        if m.CapacityOfOneTechnologyUnit[r, t, y] != 0:
            return (
                m.CapacityOfOneTechnologyUnit[r, t, y]
                * m.NumberOfNewTechnologyUnits[r, t, y]
                == m.NewCapacity[r, t, y]
            )
        return Constraint.Skip
    model.TotalNewCapacity_2 = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=TotalNewCapacity_2_rule,
    )

    # ConstraintCapacity: en cada (r,l,t,y) la suma de RateOfActivity por modo <=
    # (capacidad nueva acumulada + residual) * CapacityFactor * CapacityToActivityUnit.
    def ConstraintCapacity_rule(m, r, l, t, y):
        return (
            sum(m.RateOfActivity[r, l, t, mo, y] for mo in m.MODE_OF_OPERATION)
            <= (
                sum(
                    m.NewCapacity[r, t, yy]
                    for yy in m.YEAR
                    if ((y - yy < m.OperationalLife[r, t]) and (y - yy >= 0))
                )
                + m.ResidualCapacity[r, t, y]
            )
            * m.CapacityFactor[r, t, l, y]
            * m.CapacityToActivityUnit[r, t]
        )
    model.ConstraintCapacity = Constraint(
        model.REGION, model.TIMESLICE, model.TECHNOLOGY, model.YEAR,
        rule=ConstraintCapacity_rule,
    )

    # ####################################################################
    #    Constraints — Capacity Adequacy B
    # ####################################################################

    def PlannedMaintenance_rule(m, r, t, y):
        return (
            sum(
                m.RateOfActivity[r, l, t, mo, y] * m.YearSplit[l, y]
                for l in m.TIMESLICE
                for mo in m.MODE_OF_OPERATION
            )
            <= sum(
                (
                    sum(
                        m.NewCapacity[r, t, yy]
                        for yy in m.YEAR
                        if ((y - yy < m.OperationalLife[r, t]) and (y - yy >= 0))
                    )
                    + m.ResidualCapacity[r, t, y]
                )
                * m.CapacityFactor[r, t, l, y]
                * m.YearSplit[l, y]
                for l in m.TIMESLICE
            )
            * m.AvailabilityFactor[r, t, y]
            * m.CapacityToActivityUnit[r, t]
        )
    model.PlannedMaintenance = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=PlannedMaintenance_rule,
    )

    # ####################################################################
    #    Constraints — Energy Balance A (por timeslice: oferta >= demanda + inputs)
    # ####################################################################

    def EnergyBalanceEachTS5_rule(m, r, l, f, y):
        return (
            sum(
                m.RateOfActivity[r, l, t, mo, y]
                * m.OutputActivityRatio[r, t, f, mo, y]
                * m.YearSplit[l, y]
                for mo in m.MODE_OF_OPERATION
                for t in m.TECHNOLOGY
            )
            >= m.Demand[r, l, f, y]
            + sum(
                m.RateOfActivity[r, l, t, mo, y]
                * m.InputActivityRatio[r, t, f, mo, y]
                * m.YearSplit[l, y]
                for mo in m.MODE_OF_OPERATION
                for t in m.TECHNOLOGY
            )
        )
    model.EnergyBalanceEachTS5 = Constraint(
        model.REGION, model.TIMESLICE, model.FUEL, model.YEAR,
        rule=EnergyBalanceEachTS5_rule,
    )

    # ####################################################################
    #    Constraints — Energy Balance B (AccumulatedAnnualDemand)
    # ####################################################################

    def EnergyBalanceEachYear4_rule(m, r, f, y):
        return (
            sum(
                m.RateOfActivity[r, l, t, mo, y]
                * m.OutputActivityRatio[r, t, f, mo, y]
                * m.YearSplit[l, y]
                for t in m.TECHNOLOGY
                for mo in m.MODE_OF_OPERATION
                for l in m.TIMESLICE
            )
            >= sum(
                m.RateOfActivity[r, l, t, mo, y]
                * m.InputActivityRatio[r, t, f, mo, y]
                * m.YearSplit[l, y]
                for mo in m.MODE_OF_OPERATION
                for t in m.TECHNOLOGY
                for l in m.TIMESLICE
            )
            + m.AccumulatedAnnualDemand[r, f, y]
        )
    model.EnergyBalanceEachYear4 = Constraint(
        model.REGION, model.FUEL, model.YEAR,
        rule=EnergyBalanceEachYear4_rule,
    )

    # ####################################################################
    #    Constraints — Capital Costs
    # ####################################################################

    def UndiscountedCapitalInvestment_rule(m, r, t, y):
        return (
            m.CapitalCost[r, t, y]
            * m.NewCapacity[r, t, y]
            * m.CapitalRecoveryFactor[r, t]
            * m.PvAnnuity[r, t]
            == m.CapitalInvestment[r, t, y]
        )
    model.UndiscountedCapitalInvestment = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=UndiscountedCapitalInvestment_rule,
    )

    def DiscountedCapitalInvestment_rule(m, r, t, y):
        return (
            m.CapitalInvestment[r, t, y]
            / ((1 + m.DiscountRate[r]) ** (y - min(m.YEAR)))
            == m.DiscountedCapitalInvestment[r, t, y]
        )
    model.DiscountedCapitalInvestment_constraint = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=DiscountedCapitalInvestment_rule,
    )

    # ####################################################################
    #    Constraints — Operating Costs
    # ####################################################################

    def OperatingCostsVariable_rule(m, r, t, y):
        return (
            sum(
                sum(
                    m.RateOfActivity[r, l, t, mo, y] * m.YearSplit[l, y]
                    for l in m.TIMESLICE
                )
                * m.VariableCost[r, t, mo, y]
                for mo in m.MODE_OF_OPERATION
            )
            == m.AnnualVariableOperatingCost[r, t, y]
        )
    model.OperatingCostsVariable = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=OperatingCostsVariable_rule,
    )

    def OperatingCostsFixedAnnual_rule(m, r, t, y):
        return (
            (
                sum(
                    m.NewCapacity[r, t, yy]
                    for yy in m.YEAR
                    if ((y - yy < m.OperationalLife[r, t]) and (y - yy >= 0))
                )
                + m.ResidualCapacity[r, t, y]
            )
            * m.FixedCost[r, t, y]
            == m.AnnualFixedOperatingCost[r, t, y]
        )
    model.OperatingCostsFixedAnnual = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=OperatingCostsFixedAnnual_rule,
    )

    def OperatingCostsTotalAnnual_rule(m, r, t, y):
        return (
            m.AnnualFixedOperatingCost[r, t, y]
            + m.AnnualVariableOperatingCost[r, t, y]
            == m.OperatingCost[r, t, y]
        )
    model.OperatingCostsTotalAnnual = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=OperatingCostsTotalAnnual_rule,
    )

    def DiscountedOperatingCostsTotalAnnual_rule(m, r, t, y):
        return (
            m.OperatingCost[r, t, y]
            / ((1 + m.DiscountRate[r]) ** (y - min(m.YEAR) + 0.5))
            == m.DiscountedOperatingCost[r, t, y]
        )
    model.DiscountedOperatingCostsTotalAnnual = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=DiscountedOperatingCostsTotalAnnual_rule,
    )

    # ####################################################################
    #    Constraints — Total Discounted Costs
    # ####################################################################

    def TotalDiscountedCostByTechnology_rule(m, r, t, y):
        return (
            m.DiscountedOperatingCost[r, t, y]
            + m.DiscountedCapitalInvestment[r, t, y]
            + m.DiscountedTechnologyEmissionsPenalty[r, t, y]
            - m.DiscountedSalvageValue[r, t, y]
            + m.DiscountedDisposalCost[r, t, y]
            - m.DiscountedRecoveryValue[r, t, y]
            == m.TotalDiscountedCostByTechnology[r, t, y]
        )
    model.TotalDiscountedCostByTechnology_constraint = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=TotalDiscountedCostByTechnology_rule,
    )

    if has_storage:
        def TotalDiscountedCost_rule(m, r, y):
            return (
                sum(m.TotalDiscountedCostByTechnology[r, t, y] for t in m.TECHNOLOGY)
                + sum(m.TotalDiscountedStorageCost[r, s, y] for s in m.STORAGE)
                == m.TotalDiscountedCost[r, y]
            )
        model.TotalDiscountedCost_constraint = Constraint(
            model.REGION, model.YEAR, rule=TotalDiscountedCost_rule,
        )
    else:
        def TotalDiscountedCost_rule(m, r, y):
            return (
                sum(m.TotalDiscountedCostByTechnology[r, t, y] for t in m.TECHNOLOGY)
                == m.TotalDiscountedCost[r, y]
            )
        model.TotalDiscountedCost_constraint = Constraint(
            model.REGION, model.YEAR, rule=TotalDiscountedCost_rule,
        )

    # ####################################################################
    #    Constraints — Total Capacity Constraints
    # ####################################################################

    def TotalAnnualMaxCapacityConstraint_rule(m, r, t, y):
        if m.TotalAnnualMaxCapacity[r, t, y] == 9999999:
            return Constraint.Skip
        return (
            sum(
                m.NewCapacity[r, t, yy]
                for yy in m.YEAR
                if ((y - yy < m.OperationalLife[r, t]) and (y - yy >= 0))
            )
            + m.ResidualCapacity[r, t, y]
            <= m.TotalAnnualMaxCapacity[r, t, y]
        )
    model.TotalAnnualMaxCapacityConstraint = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=TotalAnnualMaxCapacityConstraint_rule,
    )

    def TotalAnnualMinCapacityConstraint_rule(m, r, t, y):
        return (
            sum(
                m.NewCapacity[r, t, yy]
                for yy in m.YEAR
                if ((y - yy < m.OperationalLife[r, t]) and (y - yy >= 0))
            )
            + m.ResidualCapacity[r, t, y]
            >= m.TotalAnnualMinCapacity[r, t, y]
        )
    model.TotalAnnualMinCapacityConstraint = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=TotalAnnualMinCapacityConstraint_rule,
    )

    # ####################################################################
    #    Constraints — Salvage Value
    # ####################################################################

    def SalvageValueAtEndOfPeriod1_rule(m, r, t, y):
        if (
            m.DepreciationMethod[r] == 1
            and ((y + m.OperationalLife[r, t] - 1) > max(m.YEAR))
            and m.DiscountRate[r] > 0
        ):
            return m.SalvageValue[r, t, y] == (
                m.CapitalCost[r, t, y]
                * m.NewCapacity[r, t, y]
                * m.CapitalRecoveryFactor[r, t]
                * m.PvAnnuity[r, t]
                * (
                    1
                    - (
                        ((1 + m.DiscountRate[r]) ** (max(m.YEAR) - y + 1) - 1)
                        / ((1 + m.DiscountRate[r]) ** m.OperationalLife[r, t] - 1)
                    )
                )
            )
        elif (
            m.DepreciationMethod[r] == 1
            and ((y + m.OperationalLife[r, t] - 1) > max(m.YEAR))
            and m.DiscountRate[r] == 0
        ) or (
            m.DepreciationMethod[r] == 2
            and (y + m.OperationalLife[r, t] - 1) > max(m.YEAR)
        ):
            return m.SalvageValue[r, t, y] == (
                m.CapitalCost[r, t, y]
                * m.NewCapacity[r, t, y]
                * m.CapitalRecoveryFactor[r, t]
                * m.PvAnnuity[r, t]
                * (1 - (max(m.YEAR) - y + 1) / m.OperationalLife[r, t])
            )
        return m.SalvageValue[r, t, y] == 0
    model.SalvageValueAtEndOfPeriod1 = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=SalvageValueAtEndOfPeriod1_rule,
    )

    def SalvageValueDiscountedToStartYear_rule(m, r, t, y):
        return m.DiscountedSalvageValue[r, t, y] == m.SalvageValue[r, t, y] / (
            (1 + m.DiscountRate[r]) ** (1 + max(m.YEAR) - min(m.YEAR))
        )
    model.SalvageValueDiscountedToStartYear = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=SalvageValueDiscountedToStartYear_rule,
    )

    # ####################################################################
    #    Constraints — New Capacity Constraints
    # ####################################################################

    def TotalAnnualMaxNewCapacityConstraint_rule(m, r, t, y):
        if m.TotalAnnualMaxCapacityInvestment[r, t, y] == 9999999:
            return Constraint.Skip
        return m.NewCapacity[r, t, y] <= m.TotalAnnualMaxCapacityInvestment[r, t, y]
    model.TotalAnnualMaxNewCapacityConstraint = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=TotalAnnualMaxNewCapacityConstraint_rule,
    )

    def TotalAnnualMinNewCapacityConstraint_rule(m, r, t, y):
        return m.NewCapacity[r, t, y] >= m.TotalAnnualMinCapacityInvestment[r, t, y]
    model.TotalAnnualMinNewCapacityConstraint = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=TotalAnnualMinNewCapacityConstraint_rule,
    )

    # ####################################################################
    #    Constraints — Annual Activity Constraints
    # ####################################################################

    def TotalAnnualTechnologyActivityUpperLimit_rule(m, r, t, y):
        if m.TotalTechnologyAnnualActivityUpperLimit[r, t, y] == 9999999:
            return Constraint.Skip
        return (
            sum(
                sum(m.RateOfActivity[r, l, t, mo, y] for mo in m.MODE_OF_OPERATION)
                * m.YearSplit[l, y]
                for l in m.TIMESLICE
            )
            <= m.TotalTechnologyAnnualActivityUpperLimit[r, t, y]
        )
    model.TotalAnnualTechnologyActivityUpperlimit = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=TotalAnnualTechnologyActivityUpperLimit_rule,
    )

    def TotalAnnualTechnologyActivityLowerLimit_rule(m, r, t, y):
        return (
            sum(
                sum(m.RateOfActivity[r, l, t, mo, y] for mo in m.MODE_OF_OPERATION)
                * m.YearSplit[l, y]
                for l in m.TIMESLICE
            )
            >= m.TotalTechnologyAnnualActivityLowerLimit[r, t, y]
        )
    model.TotalAnnualTechnologyActivityLowerlimit = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=TotalAnnualTechnologyActivityLowerLimit_rule,
    )

    # ####################################################################
    #    Constraints — Total Activity Constraints
    # ####################################################################

    def TotalModelHorizonTechnologyActivity_rule(m, r, t):
        return (
            sum(
                sum(
                    sum(m.RateOfActivity[r, l, t, mo, y] for mo in m.MODE_OF_OPERATION)
                    * m.YearSplit[l, y]
                    for l in m.TIMESLICE
                )
                for y in m.YEAR
            )
            == m.TotalTechnologyModelPeriodActivity[r, t]
        )
    model.TotalModelHorizonTechnologyActivity = Constraint(
        model.REGION, model.TECHNOLOGY,
        rule=TotalModelHorizonTechnologyActivity_rule,
    )

    def TotalModelHorizonTechnologyActivityUpperLimit_rule(m, r, t):
        if m.TotalTechnologyModelPeriodActivityUpperLimit[r, t] == 9999999:
            return Constraint.Skip
        return (
            m.TotalTechnologyModelPeriodActivity[r, t]
            <= m.TotalTechnologyModelPeriodActivityUpperLimit[r, t]
        )
    model.TotalModelHorizonTechnologyActivityUpperLimit = Constraint(
        model.REGION, model.TECHNOLOGY,
        rule=TotalModelHorizonTechnologyActivityUpperLimit_rule,
    )

    def TotalModelHorizonTechnologyActivityLowerLimit_rule(m, r, t):
        return (
            m.TotalTechnologyModelPeriodActivity[r, t]
            >= m.TotalTechnologyModelPeriodActivityLowerLimit[r, t]
        )
    model.TotalModelHorizonTechnologyActivityLowerLimit = Constraint(
        model.REGION, model.TECHNOLOGY,
        rule=TotalModelHorizonTechnologyActivityLowerLimit_rule,
    )

    # ####################################################################
    #    Constraints — Emissions Accounting (emisión por tecnología/modo, límites anuales y periodo)
    # ####################################################################

    def AnnualEmissionProductionByMode_rule(m, r, t, e, mo, y):
        if m.EmissionActivityRatio[r, t, e, mo, y] != 0:
            return (
                m.EmissionActivityRatio[r, t, e, mo, y]
                * sum(m.RateOfActivity[r, l, t, mo, y] * m.YearSplit[l, y] for l in m.TIMESLICE)
                == m.AnnualTechnologyEmissionByMode[r, t, e, mo, y]
            )
        return m.AnnualTechnologyEmissionByMode[r, t, e, mo, y] == 0
    model.AnnualEmissionProductionByMode = Constraint(
        model.REGION, model.TECHNOLOGY, model.EMISSION, model.MODE_OF_OPERATION, model.YEAR,
        rule=AnnualEmissionProductionByMode_rule,
    )

    def AnnualEmissionProduction_rule(m, r, t, e, y):
        return (
            sum(m.AnnualTechnologyEmissionByMode[r, t, e, mo, y] for mo in m.MODE_OF_OPERATION)
            == m.AnnualTechnologyEmission[r, t, e, y]
        )
    model.AnnualEmissionProduction = Constraint(
        model.REGION, model.TECHNOLOGY, model.EMISSION, model.YEAR,
        rule=AnnualEmissionProduction_rule,
    )

    def EmissionPenaltyByTechAndEmission_rule(m, r, t, e, y):
        return (
            m.AnnualTechnologyEmission[r, t, e, y] * m.EmissionsPenalty[r, e, y]
            == m.AnnualTechnologyEmissionPenaltyByEmission[r, t, e, y]
        )
    model.EmissionPenaltyByTechAndEmission = Constraint(
        model.REGION, model.TECHNOLOGY, model.EMISSION, model.YEAR,
        rule=EmissionPenaltyByTechAndEmission_rule,
    )

    def EmissionsPenaltyByTechnology_rule(m, r, t, y):
        return (
            sum(m.AnnualTechnologyEmissionPenaltyByEmission[r, t, e, y] for e in m.EMISSION)
            == m.AnnualTechnologyEmissionsPenalty[r, t, y]
        )
    model.EmissionsPenaltyByTechnology = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=EmissionsPenaltyByTechnology_rule,
    )

    def DiscountedEmissionsPenaltyByTechnology_rule(m, r, t, y):
        return (
            m.AnnualTechnologyEmissionsPenalty[r, t, y]
            / ((1 + m.DiscountRate[r]) ** (y - min(m.YEAR) + 0.5))
            == m.DiscountedTechnologyEmissionsPenalty[r, t, y]
        )
    model.DiscountedEmissionsPenaltyByTechnology = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=DiscountedEmissionsPenaltyByTechnology_rule,
    )

    def EmissionsAccounting1_rule(m, r, e, y):
        return (
            sum(m.AnnualTechnologyEmission[r, t, e, y] for t in m.TECHNOLOGY)
            == m.AnnualEmissions[r, e, y]
        )
    model.EmissionsAccounting1 = Constraint(
        model.REGION, model.EMISSION, model.YEAR,
        rule=EmissionsAccounting1_rule,
    )

    def EmissionsAccounting2_rule(m, r, e):
        return (
            sum(m.AnnualEmissions[r, e, y] for y in m.YEAR)
            == m.ModelPeriodEmissions[r, e] - m.ModelPeriodExogenousEmission[r, e]
        )
    model.EmissionsAccounting2 = Constraint(
        model.REGION, model.EMISSION,
        rule=EmissionsAccounting2_rule,
    )

    def AnnualEmissionsLimit_rule(m, r, e, y):
        if m.AnnualEmissionLimit[r, e, y] == 9999999:
            return Constraint.Skip
        return (
            m.AnnualEmissions[r, e, y] + m.AnnualExogenousEmission[r, e, y]
            <= m.AnnualEmissionLimit[r, e, y]
        )
    model.AnnualEmissionsLimit = Constraint(
        model.REGION, model.EMISSION, model.YEAR,
        rule=AnnualEmissionsLimit_rule,
    )

    def ModelPeriodEmissionsLimit_rule(m, r, e):
        if m.ModelPeriodEmissionLimit[r, e] == 9999999:
            return Constraint.Skip
        return m.ModelPeriodEmissions[r, e] <= m.ModelPeriodEmissionLimit[r, e]
    model.ModelPeriodEmissionsLimit = Constraint(
        model.REGION, model.EMISSION,
        rule=ModelPeriodEmissionsLimit_rule,
    )

    # ####################################################################
    #    Constraints — Reserve Margin (capacidad en reserva >= demanda * factor por timeslice)
    # ####################################################################

    def ReserveMargin_TechnologiesIncluded_rule(m, r, y):
        return (
            sum(
                (
                    sum(
                        m.NewCapacity[r, t, yy]
                        for yy in m.YEAR
                        if ((y - yy < m.OperationalLife[r, t]) and (y - yy >= 0))
                    )
                    + m.ResidualCapacity[r, t, y]
                )
                * m.ReserveMarginTagTechnology[r, t, y]
                * m.CapacityToActivityUnit[r, t]
                for t in m.TECHNOLOGY
            )
            == m.TotalCapacityInReserveMargin[r, y]
        )
    model.ReserveMargin_TechnologiesIncluded = Constraint(
        model.REGION, model.YEAR,
        rule=ReserveMargin_TechnologiesIncluded_rule,
    )

    def ReserveMargin_FuelsIncluded_rule(m, r, l, y):
        return (
            sum(
                sum(
                    m.RateOfActivity[r, l, t, mo, y] * m.OutputActivityRatio[r, t, f, mo, y]
                    for t in m.TECHNOLOGY
                    for mo in m.MODE_OF_OPERATION
                )
                * m.ReserveMarginTagFuel[r, f, y]
                for f in m.FUEL
            )
            == m.DemandNeedingReserveMargin[r, l, y]
        )
    model.ReserveMargin_FuelsIncluded = Constraint(
        model.REGION, model.TIMESLICE, model.YEAR,
        rule=ReserveMargin_FuelsIncluded_rule,
    )

    def ReserveMarginConstraint_rule(m, r, l, y):
        return (
            m.DemandNeedingReserveMargin[r, l, y] * m.ReserveMargin[r, y]
            <= m.TotalCapacityInReserveMargin[r, y]
        )
    model.ReserveMarginConstraint = Constraint(
        model.REGION, model.TIMESLICE, model.YEAR,
        rule=ReserveMarginConstraint_rule,
    )

    # ####################################################################
    #    Constraints — MUIO
    # ####################################################################

    def LU1_rule(m, r, t, mo, y):
        if m.TechnologyActivityByModeUpperLimit[r, t, mo, y] == 0:
            return Constraint.Skip
        return (
            sum(m.RateOfActivity[r, l, t, mo, y] * m.YearSplit[l, y] for l in m.TIMESLICE)
            <= m.TechnologyActivityByModeUpperLimit[r, t, mo, y]
        )
    model.LU1_TechnologyActivityByModeUL = Constraint(
        model.REGION, model.TECHNOLOGY, model.MODE_OF_OPERATION, model.YEAR,
        rule=LU1_rule,
    )

    def LU2_rule(m, r, t, mo, y):
        return (
            sum(m.RateOfActivity[r, l, t, mo, y] * m.YearSplit[l, y] for l in m.TIMESLICE)
            >= m.TechnologyActivityByModeLowerLimit[r, t, mo, y]
        )
    model.LU2_TechnologyActivityByModeLL = Constraint(
        model.REGION, model.TECHNOLOGY, model.MODE_OF_OPERATION, model.YEAR,
        rule=LU2_rule,
    )

    def LU3_rule(m, r, t, mo, y, yy):
        if (y - yy != 1) or (m.TechnologyActivityIncreaseByModeLimit[r, t, mo, yy] == 0):
            return Constraint.Skip
        return (
            sum(m.RateOfActivity[r, l, t, mo, y] * m.YearSplit[l, y] for l in m.TIMESLICE)
            <= (1 + m.TechnologyActivityIncreaseByModeLimit[r, t, mo, yy])
            * sum(m.RateOfActivity[r, l, t, mo, yy] * m.YearSplit[l, yy] for l in m.TIMESLICE)
        )
    model.LU3_TechnologyActivityIncreaseByMode = Constraint(
        model.REGION, model.TECHNOLOGY, model.MODE_OF_OPERATION, model.YEAR, model.YEAR,
        rule=LU3_rule,
    )

    def LU4_rule(m, r, t, mo, y, yy):
        if (y - yy != 1) or (m.TechnologyActivityDecreaseByModeLimit[r, t, mo, yy] == 0):
            return Constraint.Skip
        return (
            sum(m.RateOfActivity[r, l, t, mo, y] * m.YearSplit[l, y] for l in m.TIMESLICE)
            >= (1 - m.TechnologyActivityDecreaseByModeLimit[r, t, mo, yy])
            * sum(m.RateOfActivity[r, l, t, mo, yy] * m.YearSplit[l, yy] for l in m.TIMESLICE)
        )
    model.LU4_TechnologyActivityDecreaseByMode = Constraint(
        model.REGION, model.TECHNOLOGY, model.MODE_OF_OPERATION, model.YEAR, model.YEAR,
        rule=LU4_rule,
    )

    # ####################################################################
    #    Constraints — UDC (User-Defined Constraints: combinación lineal de capacidad/actividad vs constante)
    # ####################################################################

    if has_udc:
        def UDC1_rule(m, r, u, y):
            if m.UDCTag[r, u] != 0:
                return Constraint.Skip
            return (
                sum(
                    m.UDCMultiplierTotalCapacity[r, t, u, y]
                    * (
                        sum(
                            m.NewCapacity[r, t, yy]
                            for yy in m.YEAR
                            if ((y - yy < m.OperationalLife[r, t]) and (y - yy >= 0))
                        )
                        + m.ResidualCapacity[r, t, y]
                    )
                    for t in m.TECHNOLOGY
                )
                + sum(
                    m.UDCMultiplierNewCapacity[r, t, u, y] * m.NewCapacity[r, t, y]
                    for t in m.TECHNOLOGY
                )
                + sum(
                    m.UDCMultiplierActivity[r, t, u, y]
                    * sum(
                        m.RateOfActivity[r, l, t, mo, y] * m.YearSplit[l, y]
                        for l in m.TIMESLICE
                        for mo in m.MODE_OF_OPERATION
                    )
                    for t in m.TECHNOLOGY
                )
                <= m.UDCConstant[r, u, y]
            )
        model.UDC1_UserDefinedConstraintInequality = Constraint(
            model.REGION, model.UDC, model.YEAR, rule=UDC1_rule,
        )

        def UDC2_rule(m, r, u, y):
            if m.UDCTag[r, u] != 1:
                return Constraint.Skip
            return (
                sum(
                    m.UDCMultiplierTotalCapacity[r, t, u, y]
                    * (
                        sum(
                            m.NewCapacity[r, t, yy]
                            for yy in m.YEAR
                            if ((y - yy < m.OperationalLife[r, t]) and (y - yy >= 0))
                        )
                        + m.ResidualCapacity[r, t, y]
                    )
                    for t in m.TECHNOLOGY
                )
                + sum(
                    m.UDCMultiplierNewCapacity[r, t, u, y] * m.NewCapacity[r, t, y]
                    for t in m.TECHNOLOGY
                )
                + sum(
                    m.UDCMultiplierActivity[r, t, u, y]
                    * sum(
                        m.RateOfActivity[r, l, t, mo, y] * m.YearSplit[l, y]
                        for l in m.TIMESLICE
                        for mo in m.MODE_OF_OPERATION
                    )
                    for t in m.TECHNOLOGY
                )
                == m.UDCConstant[r, u, y]
            )
        model.UDC2_UserDefinedConstraintEquality = Constraint(
            model.REGION, model.UDC, model.YEAR, rule=UDC2_rule,
        )

    # ####################################################################
    #    Constraints — Disposal / Recovery (Max B/C)
    # ####################################################################

    def DisposalCostAtEndOfPeriod1_rule(m, r, t, y):
        return m.DisposalCost[r, t, y] == (
            m.DisposalCostPerCapacity[r, t] * m.CapitalCost[r, t, y] * m.NewCapacity[r, t, y]
        )
    model.DisposalCostAtEndOfPeriod1 = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=DisposalCostAtEndOfPeriod1_rule,
    )

    def DisposalCostDiscounted_rule(m, r, t, y):
        return m.DiscountedDisposalCost[r, t, y] == m.DisposalCost[r, t, y] / (
            (1 + m.DiscountRate[r]) ** (y + m.OperationalLife[r, t] - min(m.YEAR))
        )
    model.DiscountedDisposalCost_constraint = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=DisposalCostDiscounted_rule,
    )

    def RecoveryValueAtEndOfPeriod_rule(m, r, t, y):
        return m.RecoveryValue[r, t, y] == (
            m.RecoveryValuePerCapacity[r, t] * m.CapitalCost[r, t, y] * m.NewCapacity[r, t, y]
        )
    model.RecoveryValueAtEndOfPeriod = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=RecoveryValueAtEndOfPeriod_rule,
    )

    def RecoveryValueDiscounted_rule(m, r, t, y):
        return m.DiscountedRecoveryValue[r, t, y] == m.RecoveryValue[r, t, y] / (
            (1 + m.DiscountRate[r]) ** (y + m.OperationalLife[r, t] - min(m.YEAR))
        )
    model.RecoveryValueDiscounted_constraint = Constraint(
        model.REGION, model.TECHNOLOGY, model.YEAR,
        rule=RecoveryValueDiscounted_rule,
    )

    # ####################################################################
    #    Constraints — Storage
    # ####################################################################

    if has_storage:
        _add_storage_constraints(model)

    return model


# ========================================================================
#  Storage constraints (separadas en helper para legibilidad)
# ========================================================================

def _add_storage_constraints(model: AbstractModel) -> None:
    """Agrega todas las restricciones de almacenamiento al modelo."""

    # --- Storage equations ---

    def RateOfStorageCharge_rule(m, r, s, ls, ld, lh, y, t, mo):
        if m.TechnologyToStorage[r, t, s, mo] > 0:
            return (
                sum(
                    m.RateOfActivity[r, l, t, mo, y]
                    * m.TechnologyToStorage[r, t, s, mo]
                    * m.Conversionls[l, ls]
                    * m.Conversionld[l, ld]
                    * m.Conversionlh[l, lh]
                    for mo in m.MODE_OF_OPERATION
                    for l in m.TIMESLICE
                    for t in m.TECHNOLOGY
                )
                == m.RateOfStorageCharge[r, s, ls, ld, lh, y]
            )
        return Constraint.Skip
    model.RateOfStorageCharge_constraint = Constraint(
        model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
        model.DAILYTIMEBRACKET, model.YEAR, model.TECHNOLOGY, model.MODE_OF_OPERATION,
        rule=RateOfStorageCharge_rule,
    )

    def RateOfStorageDischarge_rule(m, r, s, ls, ld, lh, y, t, mo):
        if m.TechnologyFromStorage[r, t, s, mo] > 0:
            return (
                sum(
                    m.RateOfActivity[r, l, t, mo, y]
                    * m.TechnologyFromStorage[r, t, s, mo]
                    * m.Conversionls[l, ls]
                    * m.Conversionld[l, ld]
                    * m.Conversionlh[l, lh]
                    for mo in m.MODE_OF_OPERATION
                    for l in m.TIMESLICE
                    for t in m.TECHNOLOGY
                )
                == m.RateOfStorageDischarge[r, s, ls, ld, lh, y]
            )
        return Constraint.Skip
    model.RateOfStorageDischarge_constraint = Constraint(
        model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
        model.DAILYTIMEBRACKET, model.YEAR, model.TECHNOLOGY, model.MODE_OF_OPERATION,
        rule=RateOfStorageDischarge_rule,
    )

    def NetChargeWithinYear_rule(m, r, s, ls, ld, lh, y):
        return (
            sum(
                (m.RateOfStorageCharge[r, s, ls, ld, lh, y]
                 - m.RateOfStorageDischarge[r, s, ls, ld, lh, y])
                * m.YearSplit[l, y]
                * m.Conversionls[l, ls]
                * m.Conversionld[l, ld]
                * m.Conversionlh[l, lh]
                for l in m.TIMESLICE
            )
            == m.NetChargeWithinYear[r, s, ls, ld, lh, y]
        )
    model.NetChargeWithinYear_constraint = Constraint(
        model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
        model.DAILYTIMEBRACKET, model.YEAR,
        rule=NetChargeWithinYear_rule,
    )

    def NetChargeWithinDay_rule(m, r, s, ls, ld, lh, y):
        return (
            (m.RateOfStorageCharge[r, s, ls, ld, lh, y]
             - m.RateOfStorageDischarge[r, s, ls, ld, lh, y])
            * m.DaySplit[lh, y]
            == m.NetChargeWithinDay[r, s, ls, ld, lh, y]
        )
    model.NetChargeWithinDay_constraint = Constraint(
        model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
        model.DAILYTIMEBRACKET, model.YEAR,
        rule=NetChargeWithinDay_rule,
    )

    # --- Storage level tracking ---

    def StorageLevelYearStart_rule(m, r, s, y):
        if y == min(m.YEAR):
            return m.StorageLevelStart[r, s] == m.StorageLevelYearStart[r, s, y]
        return (
            m.StorageLevelYearStart[r, s, y - 1]
            + sum(
                m.NetChargeWithinYear[r, s, ls, ld, lh, y - 1]
                for ls in m.SEASON for ld in m.DAYTYPE for lh in m.DAILYTIMEBRACKET
            )
            == m.StorageLevelYearStart[r, s, y]
        )
    model.StorageLevelYearStart_constraint = Constraint(
        model.REGION, model.STORAGE, model.YEAR,
        rule=StorageLevelYearStart_rule,
    )

    def StorageLevelYearFinish_rule(m, r, s, y):
        if y < max(m.YEAR):
            return m.StorageLevelYearStart[r, s, y + 1] == m.StorageLevelYearFinish[r, s, y]
        return (
            m.StorageLevelYearStart[r, s, y]
            + sum(
                m.NetChargeWithinYear[r, s, ls, ld, lh, y - 1]
                for ls in m.SEASON for ld in m.DAYTYPE for lh in m.DAILYTIMEBRACKET
            )
            == m.StorageLevelYearFinish[r, s, y]
        )
    model.StorageLevelYearFinish_constraint = Constraint(
        model.REGION, model.STORAGE, model.YEAR,
        rule=StorageLevelYearFinish_rule,
    )

    def StorageLevelSeasonStart_rule(m, r, s, ls, y):
        if ls == min(m.SEASON):
            return m.StorageLevelYearStart[r, s, y] == m.StorageLevelSeasonStart[r, s, ls, y]
        return (
            m.StorageLevelSeasonStart[r, s, ls - 1, y]
            + sum(
                m.NetChargeWithinYear[r, s, ls - 1, ld, lh, y]
                for ld in m.DAYTYPE for lh in m.DAILYTIMEBRACKET
            )
            == m.StorageLevelSeasonStart[r, s, ls, y]
        )
    model.StorageLevelSeasonStart_constraint = Constraint(
        model.REGION, model.STORAGE, model.SEASON, model.YEAR,
        rule=StorageLevelSeasonStart_rule,
    )

    def StorageLevelDayTypeStart_rule(m, r, s, ls, ld, y):
        if ld == min(m.DAYTYPE):
            return (
                m.StorageLevelSeasonStart[r, s, ls, y]
                == m.StorageLevelDayTypeStart[r, s, ls, ld, y]
            )
        return (
            m.StorageLevelDayTypeStart[r, s, ls, ld - 1, y]
            + sum(
                m.NetChargeWithinDay[r, s, ls, ld - 1, lh, y]
                for lh in m.DAILYTIMEBRACKET
            )
            == m.StorageLevelDayTypeStart[r, s, ls, ld, y]
        )
    model.StorageLevelDayTypeStart_constraint = Constraint(
        model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE, model.YEAR,
        rule=StorageLevelDayTypeStart_rule,
    )

    def StorageLevelDayTypeFinish_rule(m, r, s, ls, ld, y):
        if ld == max(m.DAYTYPE):
            if ls == max(m.SEASON):
                return (
                    m.StorageLevelYearFinish[r, s, y]
                    == m.StorageLevelDayTypeFinish[r, s, ls, ld, y]
                )
            return (
                m.StorageLevelSeasonStart[r, s, ls + 1, y]
                == m.StorageLevelDayTypeFinish[r, s, ls, ld, y]
            )
        return (
            m.StorageLevelDayTypeFinish[r, s, ls, ld + 1, y]
            - sum(
                m.NetChargeWithinDay[r, s, ls, ld + 1, lh, y]
                for lh in m.DAILYTIMEBRACKET
            )
            * m.DaysInDayType[ls, ld + 1, y]
            == m.StorageLevelDayTypeFinish[r, s, ls, ld, y]
        )
    model.StorageLevelDayTypeFinish_constraint = Constraint(
        model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE, model.YEAR,
        rule=StorageLevelDayTypeFinish_rule,
    )

    # --- Storage bounds ---

    def LowerLimit_1TimeBracket1InstanceOfDayType1week_rule(m, r, s, ls, ld, lh, y):
        return (
            0
            <= (
                m.StorageLevelDayTypeStart[r, s, ls, ld, y]
                + sum(
                    m.NetChargeWithinDay[r, s, ls, ld, lhlh, y]
                    for lhlh in m.DAILYTIMEBRACKET if (lh - lhlh > 0)
                )
            )
            - m.StorageLowerLimit[r, s, y]
        )
    model.LowerLimit_1TimeBracket1InstanceOfDayType1week_constraint = Constraint(
        model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
        model.DAILYTIMEBRACKET, model.YEAR,
        rule=LowerLimit_1TimeBracket1InstanceOfDayType1week_rule,
    )

    def LowerLimit_EndDaylyTimeBracketLastInstanceOfDayType1Week_rule(m, r, s, ls, ld, lh, y):
        if ld > min(m.DAYTYPE):
            return (
                0
                <= (
                    m.StorageLevelDayTypeStart[r, s, ls, ld, y]
                    - sum(
                        m.NetChargeWithinDay[r, s, ls, ld - 1, lhlh, y]
                        for lhlh in m.DAILYTIMEBRACKET if (lh - lhlh < 0)
                    )
                )
                - m.StorageLowerLimit[r, s, y]
            )
        return Constraint.Skip
    model.LowerLimit_EndDaylyTimeBracketLastInstanceOfDayType1Week_constraint = Constraint(
        model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
        model.DAILYTIMEBRACKET, model.YEAR,
        rule=LowerLimit_EndDaylyTimeBracketLastInstanceOfDayType1Week_rule,
    )

    def LowerLimit_EndDaylyTimeBracketLastInstanceOfDayTypeLastWeek_rule(m, r, s, ls, ld, lh, y):
        return (
            0
            <= (
                m.StorageLevelDayTypeFinish[r, s, ls, ld, y]
                - sum(
                    m.NetChargeWithinDay[r, s, ls, ld, lhlh, y]
                    for lhlh in m.DAILYTIMEBRACKET if (lh - lhlh < 0)
                )
            )
            - m.StorageLowerLimit[r, s, y]
        )
    model.LowerLimit_EndDaylyTimeBracketLastInstanceOfDayTypeLastWeek_constraint = Constraint(
        model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
        model.DAILYTIMEBRACKET, model.YEAR,
        rule=LowerLimit_EndDaylyTimeBracketLastInstanceOfDayTypeLastWeek_rule,
    )

    def LowerLimit_1TimeBracket1InstanceOfDayTypeLastweek_rule(m, r, s, ls, ld, lh, y):
        if ld > min(m.DAYTYPE):
            return (
                0
                <= (
                    m.StorageLevelDayTypeFinish[r, s, ls, ld - 1, y]
                    + sum(
                        m.NetChargeWithinDay[r, s, ls, ld, lhlh, y]
                        for lhlh in m.DAILYTIMEBRACKET if (lh - lhlh > 0)
                    )
                )
                - m.StorageLowerLimit[r, s, y]
            )
        return Constraint.Skip
    model.LowerLimit_1TimeBracket1InstanceOfDayTypeLastweek_constraint = Constraint(
        model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
        model.DAILYTIMEBRACKET, model.YEAR,
        rule=LowerLimit_1TimeBracket1InstanceOfDayTypeLastweek_rule,
    )

    def UpperLimit_1TimeBracket1InstanceOfDayType1week_rule(m, r, s, ls, ld, lh, y):
        return (
            (
                m.StorageLevelDayTypeStart[r, s, ls, ld, y]
                + sum(
                    m.NetChargeWithinDay[r, s, ls, ld, lhlh, y]
                    for lhlh in m.DAILYTIMEBRACKET if (lh - lhlh > 0)
                )
            )
            - m.StorageUpperLimit[r, s, y]
            <= 0
        )
    model.UpperLimit_1TimeBracket1InstanceOfDayType1week_constraint = Constraint(
        model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
        model.DAILYTIMEBRACKET, model.YEAR,
        rule=UpperLimit_1TimeBracket1InstanceOfDayType1week_rule,
    )

    def UpperLimit_EndDaylyTimeBracketLastInstanceOfDayType1Week_rule(m, r, s, ls, ld, lh, y):
        if ld > min(m.DAYTYPE):
            return (
                m.StorageLevelDayTypeStart[r, s, ls, ld, y]
                - sum(
                    m.NetChargeWithinDay[r, s, ls, ld - 1, lhlh, y]
                    for lhlh in m.DAILYTIMEBRACKET if (lh - lhlh < 0)
                )
            ) - m.StorageUpperLimit[r, s, y] <= 0
        return Constraint.Skip
    model.UpperLimit_EndDaylyTimeBracketLastInstanceOfDayType1Week_constraint = Constraint(
        model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
        model.DAILYTIMEBRACKET, model.YEAR,
        rule=UpperLimit_EndDaylyTimeBracketLastInstanceOfDayType1Week_rule,
    )

    def UpperLimit_EndDaylyTimeBracketLastInstanceOfDayTypeLastWeek_rule(m, r, s, ls, ld, lh, y):
        return (
            m.StorageLevelDayTypeFinish[r, s, ls, ld, y]
            - sum(
                m.NetChargeWithinDay[r, s, ls, ld, lhlh, y]
                for lhlh in m.DAILYTIMEBRACKET if (lh - lhlh < 0)
            )
        ) - m.StorageUpperLimit[r, s, y] <= 0
    model.UpperLimit_EndDaylyTimeBracketLastInstanceOfDayTypeLastWeek_constraint = Constraint(
        model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
        model.DAILYTIMEBRACKET, model.YEAR,
        rule=UpperLimit_EndDaylyTimeBracketLastInstanceOfDayTypeLastWeek_rule,
    )

    def UpperLimit_1TimeBracket1InstanceOfDayTypeLastweek_rule(m, r, s, ls, ld, lh, y):
        if ld > min(m.DAYTYPE):
            return (
                0
                >= (
                    m.StorageLevelDayTypeFinish[r, s, ls, ld - 1, y]
                    + sum(
                        m.NetChargeWithinDay[r, s, ls, ld, lhlh, y]
                        for lhlh in m.DAILYTIMEBRACKET if (lh - lhlh > 0)
                    )
                )
                - m.StorageUpperLimit[r, s, y]
            )
        return Constraint.Skip
    model.UpperLimit_1TimeBracket1InstanceOfDayTypeLastweek_constraint = Constraint(
        model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
        model.DAILYTIMEBRACKET, model.YEAR,
        rule=UpperLimit_1TimeBracket1InstanceOfDayTypeLastweek_rule,
    )

    # --- Charge/discharge rate limits ---

    def MaxChargeConstraint_rule(m, r, s, ls, ld, lh, y):
        return m.RateOfStorageCharge[r, s, ls, ld, lh, y] <= m.StorageMaxChargeRate[r, s]
    model.MaxChargeConstraint_constraint = Constraint(
        model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
        model.DAILYTIMEBRACKET, model.YEAR,
        rule=MaxChargeConstraint_rule,
    )

    def MaxDischargeConstraint_rule(m, r, s, ls, ld, lh, y):
        return m.RateOfStorageDischarge[r, s, ls, ld, lh, y] <= m.StorageMaxDischargeRate[r, s]
    model.MaxDischargeConstraint_constraint = Constraint(
        model.REGION, model.STORAGE, model.SEASON, model.DAYTYPE,
        model.DAILYTIMEBRACKET, model.YEAR,
        rule=MaxDischargeConstraint_rule,
    )

    # --- Storage investments ---

    def StorageUpperLimit_rule(m, r, s, y):
        return (
            m.AccumulatedNewStorageCapacity[r, s, y]
            + m.ResidualStorageCapacity[r, s, y]
            == m.StorageUpperLimit[r, s, y]
        )
    model.StorageUpperLimit_constraint = Constraint(
        model.REGION, model.STORAGE, model.YEAR, rule=StorageUpperLimit_rule,
    )

    def StorageLowerLimit_rule(m, r, s, y):
        return (
            m.MinStorageCharge[r, s, y] * m.StorageUpperLimit[r, s, y]
            == m.StorageLowerLimit[r, s, y]
        )
    model.StorageLowerLimit_constraint = Constraint(
        model.REGION, model.STORAGE, model.YEAR, rule=StorageLowerLimit_rule,
    )

    def TotalNewStorage_rule(m, r, s, y):
        return (
            sum(
                m.NewStorageCapacity[r, s, yy]
                for yy in m.YEAR
                if ((y - yy < m.OperationalLifeStorage[r, s]) and (y - yy >= 0))
            )
            == m.AccumulatedNewStorageCapacity[r, s, y]
        )
    model.TotalNewStorage_constraint = Constraint(
        model.REGION, model.STORAGE, model.YEAR, rule=TotalNewStorage_rule,
    )

    def UndiscountedCapitalInvestmentStorage_rule(m, r, s, y):
        return (
            m.CapitalCostStorage[r, s, y] * m.NewStorageCapacity[r, s, y]
            == m.CapitalInvestmentStorage[r, s, y]
        )
    model.UndiscountedCapitalInvestmentStorage_constraint = Constraint(
        model.REGION, model.STORAGE, model.YEAR,
        rule=UndiscountedCapitalInvestmentStorage_rule,
    )

    def DiscountingCapitalInvestmentStorage_rule(m, r, s, y):
        return (
            m.CapitalInvestmentStorage[r, s, y]
            / ((1 + m.DiscountRate[r]) ** (y - min(m.YEAR)))
            == m.DiscountedCapitalInvestmentStorage[r, s, y]
        )
    model.DiscountingCapitalInvestmentStorage_constraint = Constraint(
        model.REGION, model.STORAGE, model.YEAR,
        rule=DiscountingCapitalInvestmentStorage_rule,
    )

    def SalvageValueStorageAtEndOfPeriod_rule(m, r, s, y):
        if (
            m.DepreciationMethod[r] == 1
            and ((y + m.OperationalLifeStorage[r, s] - 1) > max(m.YEAR))
            and m.DiscountRate[r] > 0
        ):
            return m.SalvageValueStorage[r, s, y] == m.CapitalInvestmentStorage[r, s, y] * (
                1 - (
                    ((1 + m.DiscountRate[r]) ** (max(m.YEAR) - y + 1) - 1)
                    / ((1 + m.DiscountRate[r]) ** m.OperationalLifeStorage[r, s] - 1)
                )
            )
        elif (
            m.DepreciationMethod[r] == 1
            and ((y + m.OperationalLifeStorage[r, s] - 1) > max(m.YEAR))
            and m.DiscountRate[r] == 0
        ) or (
            m.DepreciationMethod[r] == 2
            and (y + m.OperationalLifeStorage[r, s] - 1) > max(m.YEAR)
        ):
            return m.SalvageValueStorage[r, s, y] == m.CapitalInvestmentStorage[r, s, y] * (
                1 - (max(m.YEAR) - y + 1) / m.OperationalLifeStorage[r, s]
            )
        return m.SalvageValueStorage[r, s, y] == 0
    model.SalvageValueStorageAtEndOfPeriod_constraint = Constraint(
        model.REGION, model.STORAGE, model.YEAR,
        rule=SalvageValueStorageAtEndOfPeriod_rule,
    )

    def SalvageValueStorageDiscountedToStartYear_rule(m, r, s, y):
        return (
            m.SalvageValueStorage[r, s, y]
            / ((1 + m.DiscountRate[r]) ** (max(m.YEAR) - min(m.YEAR) + 1))
            == m.DiscountedSalvageValueStorage[r, s, y]
        )
    model.SalvageValueDiscountedToStartYear_constraint = Constraint(
        model.REGION, model.STORAGE, model.YEAR,
        rule=SalvageValueStorageDiscountedToStartYear_rule,
    )

    def TotalDiscountedCostByStorage_rule(m, r, s, y):
        return (
            m.DiscountedCapitalInvestmentStorage[r, s, y]
            - m.DiscountedSalvageValueStorage[r, s, y]
            == m.TotalDiscountedStorageCost[r, s, y]
        )
    model.TotalDiscountedCostByStorage_constraint = Constraint(
        model.REGION, model.STORAGE, model.YEAR,
        rule=TotalDiscountedCostByStorage_rule,
    )

In [7]:
from pyomo.environ import Set, Param, Var, Constraint, Objective

# Detectar has_storage / has_udc por presencia de CSVs no vacios
def _csv_has_data(path):
    return path.is_file() and path.stat().st_size > 7

has_storage = _csv_has_data(csv_dir / "STORAGE.csv")
has_udc = _csv_has_data(csv_dir / "UDC.csv")
print(f"has_storage = {has_storage}")
print(f"has_udc     = {has_udc}")

model = create_abstract_model(has_storage=has_storage, has_udc=has_udc)

def _count(ct):
    return sum(1 for _ in model.component_objects(ct, active=True))

print(f"\nModelo abstracto:")
print(f"  Sets        : {_count(Set)}")
print(f"  Params      : {_count(Param)}")
print(f"  Vars        : {_count(Var)}")
print(f"  Constraints : {_count(Constraint)}")
print(f"  Objectives  : {_count(Objective)}")

has_storage = False
has_udc     = False

Modelo abstracto:
  Sets        : 9
  Params      : 50
  Vars        : 30
  Constraints : 44
  Objectives  : 1


## 5. Construir la instancia (build_instance inline)

Usa `pyomo.DataPortal` para cargar sets y parámetros desde los CSVs y luego `model.create_instance(data)`.

Sets cargados solo si su CSV existe y **no está vacío**. Parámetros cargados solo si tienen al menos una fila — sino se usa el default declarado en el AbstractModel.

In [8]:
import time
from pyomo.environ import DataPortal


def _load_set(data, p, fname, set_name):
    fpath = os.path.join(p, fname)
    if not os.path.exists(fpath):
        return
    with open(fpath, encoding="utf-8") as f:
        f.readline()
        first = f.readline().strip()
    if not first:
        return
    data.load(filename=fpath, set=set_name)


def _load_param(data, p, fname, param_name, index):
    fpath = os.path.join(p, fname)
    if not os.path.exists(fpath):
        return
    with open(fpath, encoding="utf-8") as f:
        f.readline()
        first = f.readline().strip()
    if not first:
        return
    data.load(filename=fpath, param=param_name, index=index)


def build_instance_inline(model, csv_dir, *, has_storage=False, has_udc=False):
    data = DataPortal()
    p = str(csv_dir)

    _load_set(data, p, "EMISSION.csv", "EMISSION")
    _load_set(data, p, "FUEL.csv", "FUEL")
    _load_set(data, p, "TIMESLICE.csv", "TIMESLICE")
    _load_set(data, p, "MODE_OF_OPERATION.csv", "MODE_OF_OPERATION")
    _load_set(data, p, "TECHNOLOGY.csv", "TECHNOLOGY")
    _load_set(data, p, "YEAR.csv", "YEAR")
    _load_set(data, p, "REGION.csv", "REGION")
    if has_storage:
        _load_set(data, p, "STORAGE.csv", "STORAGE")
        _load_set(data, p, "SEASON.csv", "SEASON")
        _load_set(data, p, "DAYTYPE.csv", "DAYTYPE")
        _load_set(data, p, "DAILYTIMEBRACKET.csv", "DAILYTIMEBRACKET")

    _load_param(data, p, "YearSplit.csv", "YearSplit", ["TIMESLICE", "YEAR"])
    _load_param(data, p, "DiscountRate.csv", "DiscountRate", ["REGION"])
    _load_param(data, p, "DepreciationMethod.csv", "DepreciationMethod", ["REGION"])
    _load_param(data, p, "CapacityToActivityUnit.csv", "CapacityToActivityUnit", ["REGION", "TECHNOLOGY"])
    _load_param(data, p, "CapacityOfOneTechnologyUnit.csv", "CapacityOfOneTechnologyUnit", ["REGION", "TECHNOLOGY", "YEAR"])
    _load_param(data, p, "OperationalLife.csv", "OperationalLife", ["REGION", "TECHNOLOGY"])

    _load_param(data, p, "TotalAnnualMaxCapacityInvestment.csv", "TotalAnnualMaxCapacityInvestment", ["REGION", "TECHNOLOGY", "YEAR"])
    _load_param(data, p, "TotalAnnualMinCapacityInvestment.csv", "TotalAnnualMinCapacityInvestment", ["REGION", "TECHNOLOGY", "YEAR"])
    _load_param(data, p, "TotalTechnologyAnnualActivityLowerLimit.csv", "TotalTechnologyAnnualActivityLowerLimit", ["REGION", "TECHNOLOGY", "YEAR"])
    _load_param(data, p, "TotalTechnologyAnnualActivityUpperLimit.csv", "TotalTechnologyAnnualActivityUpperLimit", ["REGION", "TECHNOLOGY", "YEAR"])
    _load_param(data, p, "TotalTechnologyModelPeriodActivityLowerLimit.csv", "TotalTechnologyModelPeriodActivityLowerLimit", ["REGION", "TECHNOLOGY"])
    _load_param(data, p, "TotalTechnologyModelPeriodActivityUpperLimit.csv", "TotalTechnologyModelPeriodActivityUpperLimit", ["REGION", "TECHNOLOGY"])

    _load_param(data, p, "CapacityFactor.csv", "CapacityFactor", ["REGION", "TECHNOLOGY", "TIMESLICE", "YEAR"])
    _load_param(data, p, "AvailabilityFactor.csv", "AvailabilityFactor", ["REGION", "TECHNOLOGY", "YEAR"])
    _load_param(data, p, "ResidualCapacity.csv", "ResidualCapacity", ["REGION", "TECHNOLOGY", "YEAR"])

    _load_param(data, p, "CapitalCost.csv", "CapitalCost", ["REGION", "TECHNOLOGY", "YEAR"])
    _load_param(data, p, "FixedCost.csv", "FixedCost", ["REGION", "TECHNOLOGY", "YEAR"])
    _load_param(data, p, "VariableCost.csv", "VariableCost", ["REGION", "TECHNOLOGY", "MODE_OF_OPERATION", "YEAR"])

    _load_param(data, p, "EmissionActivityRatio.csv", "EmissionActivityRatio", ["REGION", "TECHNOLOGY", "EMISSION", "MODE_OF_OPERATION", "YEAR"])
    _load_param(data, p, "EmissionsPenalty.csv", "EmissionsPenalty", ["REGION", "EMISSION", "YEAR"])
    _load_param(data, p, "ModelPeriodEmissionLimit.csv", "ModelPeriodEmissionLimit", ["REGION", "EMISSION"])
    _load_param(data, p, "ModelPeriodExogenousEmission.csv", "ModelPeriodExogenousEmission", ["REGION", "EMISSION"])
    _load_param(data, p, "AnnualExogenousEmission.csv", "AnnualExogenousEmission", ["REGION", "EMISSION", "YEAR"])
    _load_param(data, p, "AnnualEmissionLimit.csv", "AnnualEmissionLimit", ["REGION", "EMISSION", "YEAR"])

    _load_param(data, p, "InputActivityRatio.csv", "InputActivityRatio", ["REGION", "TECHNOLOGY", "FUEL", "MODE_OF_OPERATION", "YEAR"])
    _load_param(data, p, "OutputActivityRatio.csv", "OutputActivityRatio", ["REGION", "TECHNOLOGY", "FUEL", "MODE_OF_OPERATION", "YEAR"])

    _load_param(data, p, "ReserveMarginTagFuel.csv", "ReserveMarginTagFuel", ["REGION", "FUEL", "YEAR"])
    _load_param(data, p, "RETagTechnology.csv", "RETagTechnology", ["REGION", "TECHNOLOGY", "YEAR"])
    _load_param(data, p, "RETagFuel.csv", "RETagFuel", ["REGION", "FUEL", "YEAR"])
    _load_param(data, p, "REMinProductionTarget.csv", "REMinProductionTarget", ["REGION", "YEAR"])
    _load_param(data, p, "ReserveMarginTagTechnology.csv", "ReserveMarginTagTechnology", ["REGION", "TECHNOLOGY", "YEAR"])
    _load_param(data, p, "ReserveMargin.csv", "ReserveMargin", ["REGION", "YEAR"])

    _load_param(data, p, "AccumulatedAnnualDemand.csv", "AccumulatedAnnualDemand", ["REGION", "FUEL", "YEAR"])
    _load_param(data, p, "SpecifiedAnnualDemand.csv", "SpecifiedAnnualDemand", ["REGION", "FUEL", "YEAR"])
    _load_param(data, p, "SpecifiedDemandProfile.csv", "SpecifiedDemandProfile", ["REGION", "FUEL", "TIMESLICE", "YEAR"])

    _load_param(data, p, "TotalAnnualMaxCapacity.csv", "TotalAnnualMaxCapacity", ["REGION", "TECHNOLOGY", "YEAR"])
    _load_param(data, p, "TotalAnnualMinCapacity.csv", "TotalAnnualMinCapacity", ["REGION", "TECHNOLOGY", "YEAR"])

    if has_udc:
        _load_set(data, p, "UDC.csv", "UDC")
        _load_param(data, p, "UDCMultiplierTotalCapacity.csv", "UDCMultiplierTotalCapacity", ["REGION", "TECHNOLOGY", "UDC", "YEAR"])
        _load_param(data, p, "UDCMultiplierNewCapacity.csv", "UDCMultiplierNewCapacity", ["REGION", "TECHNOLOGY", "UDC", "YEAR"])
        _load_param(data, p, "UDCMultiplierActivity.csv", "UDCMultiplierActivity", ["REGION", "TECHNOLOGY", "UDC", "YEAR"])
        _load_param(data, p, "UDCConstant.csv", "UDCConstant", ["REGION", "UDC", "YEAR"])
        _load_param(data, p, "UDCTag.csv", "UDCTag", ["REGION", "UDC"])

    return model.create_instance(data)


print("Construyendo instancia desde CSVs...")
t0 = time.perf_counter()
instance = build_instance_inline(model, csv_dir, has_storage=has_storage, has_udc=has_udc)
t_build = time.perf_counter() - t0
print(f"Instancia construida en {t_build:.2f} s")

n_vars = sum(len(v) for v in instance.component_objects(Var, active=True))
n_cons = sum(len(c) for c in instance.component_objects(Constraint, active=True))
print(f"  # variables    : {n_vars}")
print(f"  # restricciones: {n_cons}")

Construyendo instancia desde CSVs...


Instancia construida en 28.13 s
  # variables    : 722307
  # restricciones: 799364


## 6. Resolver (Gurobi → HiGHS → GLPK)

Prioridad: **Gurobi** (rápido, requiere licencia), **HiGHS** (gratis, rápido), **GLPK** (gratis, lento).

Cada solver usa sus defaults. Si Gurobi falla por licencia/timeout, cae automáticamente al siguiente.

In [9]:
import time, gc
import pyomo.environ as pyo
from pyomo.opt import SolverFactory


def _gurobi_lightweight_available() -> bool:
    """Sin crear Env: solo verifica que gurobipy se pueda importar."""
    try:
        import gurobipy  # noqa
    except Exception:
        return False
    return True


def _is_available(name: str) -> bool:
    if name == "gurobi":
        return _gurobi_lightweight_available()
    try:
        s = SolverFactory(name)
        return bool(s.available(exception_flag=False))
    except Exception:
        return False


def _dispose_gurobi_env():
    try:
        import gurobipy as gp
        if hasattr(gp, "disposeDefaultEnv"):
            gp.disposeDefaultEnv()
    except Exception:
        pass
    gc.collect()


# Prioridad: Gurobi -> HiGHS -> GLPK
SOLVERS_PRIORITY = ["gurobi", "appsi_highs", "glpk"]

obj_val = None
results = None
solver_name_used = None
t_solve = None

for sname in SOLVERS_PRIORITY:
    if not _is_available(sname):
        print(f"  {sname:<14s}: NO disponible, salteando")
        continue
    print(f"\n=== Intentando {sname} ===")
    try:
        s = SolverFactory(sname)
        t0 = time.perf_counter()
        res = s.solve(instance, tee=True, load_solutions=False)
        t_solve = time.perf_counter() - t0
        raw_status = str(res.solver.termination_condition)
        print(f"\n  status      : {res.solver.status}")
        print(f"  termination : {raw_status}")
        print(f"  elapsed     : {t_solve:.2f} s")
        if "optimal" in raw_status.lower():
            instance.solutions.load_from(res)
            for o in instance.component_objects(pyo.Objective, active=True):
                obj_val = float(pyo.value(o))
                break
            print(f"  objective   : {obj_val:.6e}")
            results = res
            solver_name_used = sname
            if sname == "gurobi":
                _dispose_gurobi_env()
            break
        else:
            print("  (status no optimal)")
            results = res
            solver_name_used = sname
            if sname == "gurobi":
                _dispose_gurobi_env()
            break  # ya tenemos un resultado (aunque sea infeasible) - parar aqui
    except Exception as e:
        print(f"  ERROR con {sname}: {type(e).__name__}: {str(e)[:200]}")
        if sname == "gurobi":
            _dispose_gurobi_env()

if obj_val is None:
    if results is not None:
        print(f"\nResultado no optimo con {solver_name_used}: {results.solver.termination_condition}")
    else:
        print("\n!! Ningun solver logro siquiera intentar la resolucion.")
else:
    print(f"\nOK: resuelto con {solver_name_used} en {t_solve:.2f} s, obj = {obj_val:.6e}")


=== Intentando gurobi ===


Set parameter WLSAccessID
Set parameter WLSSecret


Set parameter LicenseID to value 2810970


WLS license 2810970 - registered to Unidad de Planeacion Minero Energetica


  ERROR con gurobi: GurobiError: Single-use license. Another Gurobi process with pid 60453 running.



=== Intentando appsi_highs ===



  status      : error
  termination : infeasible
  elapsed     : 117.95 s
  (status no optimal)

Resultado no optimo con appsi_highs: infeasible


## 7. Diagnóstico de infactibilidad

Cuando el solver reporta `infeasible`, este bloque inspecciona la instancia para detectar:

- **Restricciones violadas**: cuyo valor `body` está fuera de `[lower, upper]` con tolerancia `1e-6`.
- **Variables con bounds conflictivos**: `LB > UB`.
- **Restricciones con bounds vacíos**: ej. `0 ≤ algo ≤ -10` o demanda exigida sin oferta posible.

Solo corre si la instancia tiene una solución cargada (incluso parcial). Si Gurobi/HiGHS no devolvieron ninguna solución, hace introspección estática (`body` evaluado con valores 0 por defecto).

In [10]:
from pyomo.core import Constraint, Var, value


def run_infeasibility_diagnostics(instance, *, top_n=15, tol=1e-6):
    print("=" * 70)
    print("DIAGNOSTICO DE INFACTIBILIDAD")
    print("=" * 70)

    constraint_violations = []
    for con in instance.component_data_objects(Constraint, active=True):
        body_val = value(con.body, exception=False)
        if body_val is None:
            continue
        lb = value(con.lower, exception=False) if con.has_lb() else None
        ub = value(con.upper, exception=False) if con.has_ub() else None

        violation = 0.0
        side = ""
        if lb is not None and body_val < lb - tol:
            violation = lb - body_val
            side = "LB"
        elif ub is not None and body_val > ub + tol:
            violation = body_val - ub
            side = "UB"

        if violation > tol:
            constraint_violations.append((con.name, body_val, lb, ub, side, violation))

    constraint_violations.sort(key=lambda x: -x[5])
    print(f"\nRestricciones violadas: {len(constraint_violations)}")
    if constraint_violations:
        print(f"\nTop {top_n}:")
        for i, (name, body, lb, ub, side, vio) in enumerate(constraint_violations[:top_n]):
            lb_txt = f"{lb:.3e}" if lb is not None else "-inf"
            ub_txt = f"{ub:.3e}" if ub is not None else "+inf"
            print(f"  {i+1:2d}. {name[:60]:<60s} body={body:.3e}  bounds=[{lb_txt}, {ub_txt}]  viol={vio:.3e} ({side})")

    var_bound_conflicts = []
    for var in instance.component_data_objects(Var, active=True):
        lb = value(var.lb, exception=False) if var.has_lb() else None
        ub = value(var.ub, exception=False) if var.has_ub() else None
        if lb is not None and ub is not None and lb > ub + tol:
            var_bound_conflicts.append((var.name, lb, ub, lb - ub))
    var_bound_conflicts.sort(key=lambda x: -x[3])

    print(f"\nVariables con LB > UB: {len(var_bound_conflicts)}")
    if var_bound_conflicts:
        print(f"\nTop {top_n}:")
        for i, (name, lb, ub, gap) in enumerate(var_bound_conflicts[:top_n]):
            print(f"  {i+1:2d}. {name[:60]:<60s} LB={lb:.3e}  UB={ub:.3e}  gap={gap:.3e}")

    # Agregar conteo por tipo de restriccion violada
    if constraint_violations:
        from collections import Counter
        prefixes = Counter()
        for name, *_ in constraint_violations:
            # Extraer prefijo antes del primer "[" o ":"
            prefix = name.split("[")[0].split(":")[0]
            prefixes[prefix] += 1
        print(f"\nViolaciones por familia de restriccion (top 15):")
        for prefix, count in prefixes.most_common(15):
            print(f"  {prefix:<50s} {count}")

    print("\nRECOMENDACIONES:")
    print("  1. Verificar demanda vs capacidad disponible (por REGION/FUEL/YEAR).")
    print("  2. Revisar ResidualCapacity y TotalAnnualMaxCapacity.")
    print("  3. Inspeccionar consistencia InputActivityRatio / OutputActivityRatio.")
    print("  4. Confirmar unidades coherentes entre fuels, actividades y capacidades.")
    print("  5. Para cada FUEL, verificar que existan tecnologias que lo produzcan.")
    print("  6. Revisar ReserveMargin: capacidad disponible >= demanda pico × margin.")
    print("=" * 70)

    return {
        "constraint_violations": constraint_violations,
        "var_bound_conflicts": var_bound_conflicts,
    }


# Correr solo si el solve no fue optimo
if results is not None and "optimal" not in str(results.solver.termination_condition).lower():
    diag = run_infeasibility_diagnostics(instance)
elif obj_val is None:
    print("Sin solucion ni resultado de solver; corriendo diagnostico estatico...")
    diag = run_infeasibility_diagnostics(instance)
else:
    print("Solucion optima encontrada; no se requiere diagnostico de infactibilidad.")
    diag = None

DIAGNOSTICO DE INFACTIBILIDAD



Restricciones violadas: 9949

Top 15:
   1. OperatingCostsFixedAnnual[RE1,DEMTRAGSLFWD,2032]             body=9.104e+03  bounds=[0.000e+00, 0.000e+00]  viol=9.104e+03 (UB)
   2. OperatingCostsFixedAnnual[RE1,DEMTRAGSLFWD,2031]             body=9.085e+03  bounds=[0.000e+00, 0.000e+00]  viol=9.085e+03 (UB)
   3. OperatingCostsFixedAnnual[RE1,DEMTRAGSLFWD,2033]             body=8.975e+03  bounds=[0.000e+00, 0.000e+00]  viol=8.975e+03 (UB)
   4. OperatingCostsFixedAnnual[RE1,DEMTRAGSLFWD,2030]             body=8.957e+03  bounds=[0.000e+00, 0.000e+00]  viol=8.957e+03 (UB)
   5. OperatingCostsFixedAnnual[RE1,DEMTRAGSLFWD,2029]             body=8.751e+03  bounds=[0.000e+00, 0.000e+00]  viol=8.751e+03 (UB)
   6. OperatingCostsFixedAnnual[RE1,DEMTRAGSLFWD,2034]             body=8.663e+03  bounds=[0.000e+00, 0.000e+00]  viol=8.663e+03 (UB)
   7. OperatingCostsFixedAnnual[RE1,DEMTRAGSLFWD,2022]             body=8.633e+03  bounds=[0.000e+00, 0.000e+00]  viol=8.633e+03 (UB)
   8. OperatingCostsFix


Variables con LB > UB: 0

Violaciones por familia de restriccion (top 15):
  OperatingCostsFixedAnnual                          5163
  TotalAnnualTechnologyActivityLowerlimit            2341
  EnergyBalanceEachYear4                             2165
  TotalAnnualMinNewCapacityConstraint                177
  TotalAnnualMinCapacityConstraint                   70
  ReserveMargin_TechnologiesIncluded                 33

RECOMENDACIONES:
  1. Verificar demanda vs capacidad disponible (por REGION/FUEL/YEAR).
  2. Revisar ResidualCapacity y TotalAnnualMaxCapacity.
  3. Inspeccionar consistencia InputActivityRatio / OutputActivityRatio.
  4. Confirmar unidades coherentes entre fuels, actividades y capacidades.
  5. Para cada FUEL, verificar que existan tecnologias que lo produzcan.
  6. Revisar ReserveMargin: capacidad disponible >= demanda pico × margin.


## 8. Inspeccionar resultados

Variables principales con valores no triviales (solo si hay solución óptima).

In [11]:
if obj_val is not None:
    print(f"Objective: {obj_val:.6e}")
    print(f"\nVariables con valor != 0 (top 15 por # entradas activas):")
    summary = []
    for v in instance.component_objects(Var, active=True):
        nz = sum(1 for k in v if v[k].value is not None and abs(v[k].value) > 1e-9)
        total = len(v)
        if nz > 0:
            summary.append((v.name, nz, total))
    summary.sort(key=lambda x: -x[1])
    for name, nz, tot in summary[:15]:
        print(f"  {name:<40s} {nz:>8d} / {tot:>8d}")

    if hasattr(instance, "NewCapacity"):
        print("\nMuestra NewCapacity (top 10 por valor):")
        nc = [(k, instance.NewCapacity[k].value)
              for k in instance.NewCapacity
              if instance.NewCapacity[k].value and instance.NewCapacity[k].value > 1e-6]
        nc.sort(key=lambda x: -x[1])
        for k, v in nc[:10]:
            print(f"  {k}: {v:.4f}")
else:
    print("No hay solucion optima para inspeccionar (ver seccion 7).")

No hay solucion optima para inspeccionar (ver seccion 7).
